# HarrisWGAN demo

In [ ]:
import os, glob
import sys
from typing import List, Tuple, Union, Dict
from pathlib import Path
from abc import ABC, abstractmethod
from datetime import datetime as dt
from pathlib import Path
import re
from operator import itemgetter
from functools import partial
import importlib
import gc
import json as js
from timeit import default_timer as timer
import random

import numpy as np
import xarray as xr
import dask
import tensorflow as tf

import multiprocessing
try:
    from multiprocessing import Pool as ThreadPool
except:
    from multiprocessing.pool import ThreadPool

repo_dir = Path("/p/home/jusers/langguth1/juwels/downscaling_maelstrom/downscaling_benchmark/")
sys.path.append(str(repo_dir.joinpath("models")))
sys.path.append(str(repo_dir.joinpath("utils")))
sys.path.append(str(repo_dir.joinpath("handle_data")))
from model_engine import ModelEngine
from other_utils import find_closest_divisor

### Data pipeline for input streams of Harris WGAN 

The input differs from the other baseline models in a way that the low-res input data is not upscaled (bi-linearly interpolated) to the target grid and that high-res static data constitues another input stream. Thus, the generator yields a dictionary which is then used to set-upthe TF data pipeline.
To-Do: 
 - [x] implement updated make_tf_dataset_allmem-method in handle_data_class.py
 - [x] adapt make_tf_dataset_dyn-method accordingly

In [ ]:
def split_in_tar(
    ds: xr.Dataset,
    predictands: List = None,
    predictors: List = None,
    static_vars: List = None,
) -> Tuple[xr.Dataset, xr.Dataset]:
    """
    Split data array with variables-dimension into input and target data for downscaling
    :param da: The unsplitted data array
    :param target_var: Name of target variable which should consttute the first channel
    :param predictands: List of selected predictand variables; parse None to use
                        all predictands (vars with suffix _tar)
    :param predictors: List of selected predictor variables; parse None to use all predictors (vars with suffix _in)
    :return: The split data array.
    """
    varnames = list(ds.data_vars)

    if predictors is None:
        invars = [var for var in varnames if var.endswith("_in")]
    else:
        assert all(
            [predictor in varnames for predictor in predictors]
        ), f"At least one predictor is not a data variable. Available variables are {*varnames,}"
        invars = list(predictors)
    if predictands is None:
        tarvars = [var for var in varnames if var.endswith("_tar")]
    else:
        assert all(
            [predictand in varnames for predictand in predictands]
        ), f"At least one predictor is not a data variable. Available variables are {*varnames,}"
        tarvars = list(predictands)

    if static_vars is None:
        ds_in, ds_tar = ds[invars], ds[tarvars]

        return ds_in, ds_tar
    else:
        assert all(
            [static_var in varnames for static_var in static_vars]
        ), f"At least ostatic high-res is not a data variable. Available variables are {*varnames,}"
        statvars = list(static_vars)

        ds_in, ds_tar, ds_stat = ds[invars], ds[tarvars], ds[statvars]

        return ds_in, ds_tar, ds_stat


def reshape_ds(ds):
    """
    Convert a xarray dataset to a data-array where the variables will constitute the last dimension (channel last)
    :param ds: the xarray dataset with dimensions (dims)
    :return da: the data-array with dimensions (dims, variables)
    """
    da = ds.to_array(dim="variables")
    da = da.transpose(..., "variables")
    return da


def make_tf_dataset_allmem(
    ds: xr.Dataset,
    batch_size: int,
    predictands: List,
    predictors: List,
    static_vars: List,
    lshuffle: bool = True,
    shuffle_samples: int = 20000,
    named_targets: bool = False,
    var_tar2in: str = None,
    lrepeat: bool = True,
    drop_remainder: bool = True,
) -> tf.data.Dataset:
    """
    Build-up TensorFlow dataset from a generator based on the xarray-data array.
    NOTE: All data is loaded into memory
    :param ds: the xarray dataset. Input variable names must carry the suffix '_in', whereas it must be '_tar' for target variables
    :param batch_size: number of samples per mini-batch
    :param predictands: List of selected predictand variables
    :param predictors: List of selected predictor variables; parse None to use all predictors (vars with suffix _in)
    :param lshuffle: flag if shuffling should be applied to dataset
    :param shuffle_samples: number of samples to load before applying shuffling
    :param named_targets: flag if target of TF dataset should be dictionary with named target variables
    :param var_tar2in: name of target variable to be added to input (used e.g. for adding high-resolved topography
                                                                        to the input)
    :param lrepeat: flag if dataset should be repeated
    :param drop_remainder: flag if samples will be dropped in case batch size is not a divisor of # data samples
    :param with_horovod: flag to trigger horovod-based distributed dataset creation
    :param lembed: flag to trigger temporal embedding (not implemented yet!)
    """

    # add time dimension to constant variables
    for var in ds.data_vars:
        if "time" not in ds[var].dims:
            ds[var] = ds[var].expand_dims({"time": ds["time"]}, axis=0)

    ds_in, ds_tar, ds_stat = split_in_tar(
        ds, predictands=predictands, predictors=predictors, static_vars=static_vars
    )

    # convert dataset to data arrays and load into memory
    da_in, da_tar, da_stat = (
        reshape_ds(ds_in).astype("float32", copy=True),
        reshape_ds(ds_tar).astype("float32", copy=True),
        reshape_ds(ds_stat).astype("float32", copy=True),
    )

    if var_tar2in is not None:
        # NOTE: * The order of the following operation must be the same as in StreamMonthlyNetCDF.getitems
        #       * The following operation order must concatenate var_tar2in by da_in to ensure
        #         that the variable appears at first place. This is required to avoid
        #         that var_tar2in becomes a predeictand when slicing takes place in tf_split
        da_in = xr.concat([da_tar.sel({"variables": var_tar2in}), da_in], "variables")

    varnames_tar = da_tar["variables"].values

    def gen_named(darr_in, darr_tar):
        # darr_in, darr_tar = darr_in.load(), darr_tar.load()
        ntimes = len(darr_in["time"])
        for t in range(ntimes):
            tar_now = darr_tar.isel({"time": t})
            yield tuple(
                (
                    darr_in.isel({"time": t}).values,
                    {
                        var: tar_now.sel({"variables": var}).values
                        for var in varnames_tar
                    },
                )
            )

    def gen_unnamed(darr_in, darr_tar):
        # darr_in, darr_tar = darr_in.load(), darr_tar.load()
        ntimes = len(darr_in["time"])
        for t in range(ntimes):
            yield tuple(
                (darr_in.isel({"time": t}).values, darr_tar.isel({"time": t}).values)
            )

    def gen_dict(darr_in, darr_tar, darr_stat):
        ntimes = len(darr_in["time"])
        for t in range(ntimes):
            yield tuple(
                (
                    {
                        "lo_res_inputs": darr_in.isel({"time": t}).values,
                        "hi_res_inputs": darr_stat.isel({"time": t}).values,
                    },
                    {"output": darr_tar.isel({"time": t}).values},
                )
            )

    if named_targets is True:
        gen_now = gen_named
    elif static_vars is not None:
        gen_now = gen_dict
    else:
        gen_now = gen_unnamed

    # create output signatures from first sample
    if static_vars is None:
        s0 = next(iter(gen_now(da_in, da_tar)))
        sample_spec_in = tf.TensorSpec(s0[0].shape, dtype=s0[0].dtype)
        if named_targets is True:
            sample_spec_tar = {
                var: tf.TensorSpec(s0[1][var].shape, dtype=s0[1][var].dtype)
                for var in varnames_tar
            }
        else:
            sample_spec_tar = tf.TensorSpec(s0[1].shape, dtype=s0[1].dtype)

        # re-instantiate the generator and build TF dataset
        gen_train = gen_now(da_in, da_tar)

    else:
        s0 = next(iter(gen_now(da_in, da_tar, da_stat)))

        sample_spec_in = {
            "lo_res_inputs": tf.TensorSpec(
                s0[0]["lo_res_inputs"].shape, dtype=s0[0]["lo_res_inputs"].dtype
            ),
            "hi_res_inputs": tf.TensorSpec(
                s0[0]["hi_res_inputs"].shape, dtype=s0[0]["hi_res_inputs"].dtype
            ),
        }

        sample_spec_tar = {
            "output": tf.TensorSpec(s0[1]["output"].shape, dtype=s0[1]["output"].dtype)
        }

        # re-instantiate the generator and build TF dataset
        gen_train = gen_now(da_in, da_tar, da_stat)

    data_iter = tf.data.Dataset.from_generator(
        lambda: gen_train, output_signature=(sample_spec_in, sample_spec_tar)
    )

    # Notes:
    # * cache is reuqired to make repeat work properly on datasets based on generators
    #   (see https://stackoverflow.com/questions/60226022/tf-data-generator-keras-repeat-does-not-work-why)
    # * repeat must be applied after shuffle to get varying mini-batches per epoch
    # * batch-size is increased to allow substepping in train_step
    if lshuffle > 1:
        data_iter = (
            data_iter.cache()
            .shuffle(shuffle_samples)
            .batch(batch_size, drop_remainder=drop_remainder)
        )
    else:
        data_iter = data_iter.cache().batch(batch_size, drop_remainder=drop_remainder)

    if lrepeat:
        data_iter = data_iter.repeat()

    # clean-up to free some memory
    # free_mem([da, da_in, da_tar, varnames_tar])
    del ds
    del ds_in
    del ds_tar
    del da_in
    del da_tar
    gc.collect()

    return data_iter

In [ ]:
def make_tf_dataset_dyn(ds_obj, batch_size: int, nepochs: int, nshuffle: int, lrepeat: bool = True, drop_remainder: bool = True) -> tf.data.Dataset:
    """
    Build TensorFlow dataset by streaming from netCDF using xarray's open_mfdatset-method.
    To fit into memory, only a subset of all netCDF-files is processed at once (nfiles2merge-parameter).
    :param ds_obj: StreamMonthlyNetCDF-object
    :param batch_size: desired mini-batch size
    :param nepochs: (effective) number of epochs for training
    :param nshuffle: number of samples to shuffle (set to 1 to disable shuffling)
    :param lrepeat: flag if dataset should be repeated
    :param drop_remainder: flag if samples will be dropped in case batch size is not a divisor of # data samples
    :return: TensorFlow dataset object that streams data from subset of many netCDF-files
    """
    tf_read_nc = lambda ind_set: tf.py_function(ds_obj.read_netcdf, [ind_set], tf.int64)
    tf_choose_data = lambda il: tf.py_function(ds_obj.choose_data, [il], tf.bool)
    tf_getdata = lambda i: tf.numpy_function(ds_obj.getitems, [i], tf.float32)
    
    mode = ds_obj.stream_mode
    
    if mode in ["hi_input", "hi_input_named_target"]:
        tf_getdata = lambda i: tf.numpy_function(ds_obj.getitems, [i], tf.float32)
        if mode == "hi_input":
            tf_split = lambda arr: (arr[..., 0:-ds_obj.n_predictands], arr[..., -ds_obj.n_predictands:])
        else:
            varnames = ds_obj.predictand_list
            tf_split = lambda arr: (arr[..., 0:-ds_obj.n_predictands],
                                    {var: arr[..., -ds_obj.n_predictands + i] for i, var in enumerate(varnames)})
    else: 
        def make_dict(darr_in, darr_stat, darr_out):
            return ({"lo_res_inputs": darr_in, "hi_res_inputs": darr_stat}, {"output": darr_out})
                                         
        tf_getdata = lambda i: tf.numpy_function(ds_obj.getitems, [i], [tf.float32, tf.float32, tf.float32])
        tf_split = lambda arr_in, arr_stat, arr_out: make_dict(arr_in, arr_stat, arr_out)

    # enable flexibility in factor for range
    n_reads = int(ds_obj.nfiles_merged*nepochs)
    if ds_obj.with_horovod:
        import horovod.tensorflow as hvd
        tfds = tf.data.Dataset.range(n_reads).shard(hvd.size(), hvd.rank()).map(tf_read_nc).prefetch(1)
    else:
        tfds = tf.data.Dataset.range(n_reads).map(tf_read_nc).prefetch(1)

    tfds = tfds.flat_map(lambda x: tf.data.Dataset.from_tensors(x).map(tf_choose_data))
    tfds = tfds.flat_map(
        lambda x: tf.data.Dataset.range(ds_obj.samples_merged).shuffle(nshuffle)
        .batch(batch_size, drop_remainder=drop_remainder).map(tf_getdata, num_parallel_calls=tf.data.AUTOTUNE))

    tfds = tfds.map(tf_split, num_parallel_calls=tf.data.AUTOTUNE)
    
    if lrepeat:
        tfds = tfds.repeat()

    return tfds

class StreamMonthlyNetCDF(object):
    def __init__(self, mode: str, datadir: Path, patt: str, nfiles_merge: Union[int, Dict], predictands: List,
                 predictors: List = None, static_predictors: List = None, sample_dim: str = "time", norm_dims: List = None,
                 norm_obj=None ,with_horovod: bool = False, seed: int = None, nworkers: int = 10):
        """
        Class object providing all methods to create a TF dataset that iterates over a set of (monthly) netCDF-files
        rather than loading all into memory. Instead, only a subset of all netCDF-files is loaded into memory.
        Furthermore, the class attributes provide key information on the handled dataset
        :param datadir: directory where set of netCDF-files are located
        :param patt: filename pattern to allow globbing for netCDF-files
        :param nfiles_merge: number of files per data subset loaded into memory (can be an integer or a dictionary like {"#GPUS=1": 33})
        :param predictands: list of predictand variables names to be obtained
        :param predictors: list of predictor variable names to be obtained, pass None
                           if all vars with suffix _in should be chosen
        :param static_predictors: list of static predictor variable names to be obtained
        :param sample_dim: name of dimension in the data over which sampling should be performed
        :param var_tar2in: predictand (target) variable that can be inputted as well
                          (e.g. static variables known a priori such as the surface topography)
        :param norm_dims: list of dimensions over which data will be normalized
        :param norm_obj: normalization object providing parameters for (de-)normalization
        :param with_horovod: flag to trigger horovod-based distributed dataset creation
        :param seed: seed for random sampling of netCDF-files
        :param nworkers: number of threads to read the netCDF-files
        """
        self.with_horovod = with_horovod
        if self.with_horovod:
            import horovod.tensorflow as hvd
        self.seed = seed
        self.stream_mode = mode
        self.data_dir = datadir
        # get file list and number of files to be merged for data subse
        self.file_list = patt
        self.nfiles = len(self.file_list)
        # get relevant data dimensions
        ds_all = xr.open_mfdataset(list(self.file_list), decode_cf=False, cache=False)  # , parallel=True)
        self.all_dims = ds_all.dims
        self.sample_dim = sample_dim
        self.nsamples = ds_all.dims[sample_dim]
        self.dataset_size = self.get_dataset_size()
        # sampling of datafiles
        self.file_list_random = random.sample(self.file_list, self.nfiles)
        self.nfiles2merge = nfiles_merge                                # number of files to be merged for data subset  
        self.nfiles_merged = int(self.nfiles / self.nfiles2merge)       # number of data subsets
        self.samples_merged = self.get_samples_per_merged_file()
        # list of files can be larger for distributed training, i.e. effective dataset size can be increased
        if self.with_horovod:
            # re-do dataset calculation since file list is potentially larger for distributed training
            self.effective_dataset_size = self.get_dataset_size(random_list=True)
        else:
            self.effective_dataset_size = self.dataset_size
        # handle selected variables
        self.varnames_list = self.get_all_varnames()
        self.predictor_list = predictors
        self.static_predictor_list = static_predictors
        self.predictand_list = predictands
        self.n_predictands, self.n_predictors = len(self.predictand_list), len(self.predictor_list)
        self.all_vars = self.predictor_list + self.predictand_list 
        if self.static_predictor_list is not None:
            self.all_vars = self.static_predictor_list + self.all_vars     # ordering important to ensure that predictors come first (cf. make_tf_dataset_allmem-method)!
            self.n_predictors += len(self.static_predictor_list) 
        self.data_xy_dim = self.get_nxy_dim(ds_all)
        # sanity check on shapes of predictors, predictands and static predictors depending on stream_mode
        self.check_data_shapes()
        # get normalization object
        t0 = timer()
        # check if normalization object is provided
        self.normalization_time = -999.
        if norm_obj is None:
            print("Start computing normalization parameters.")
            self.data_norm = ZScore(norm_dims)  # TO-DO: Allow for arbitrary normalization
            self.norm_params = self.data_norm.get_required_stats(ds_all)
            self.normalization_time = timer() - t0
        else:
            self.data_norm = norm_obj
            self.norm_params = norm_obj.norm_stats

        # initialize data loading
        self.data_loaded = [xr.Dataset, xr.Dataset]        # two datasets will be cached
        self.iload_next, self.iuse_next = 0, 0
        self.reading_times = []
        self.ds_proc_size = 0.
        self.data_now = None
        if not nworkers:
            nworkers = min((multiprocessing.cpu_count(), self.nfiles2merge))
        self.pool = ThreadPool(nworkers)
        
        del ds_all
        gc.collect()

    @property
    def stream_mode(self):
        return self._stream_mode
    
    @stream_mode.setter
    def stream_mode(self, mode):
        known_modes = ["hi_input", "hi_input_named_target", "lo_input"]
        if mode not in known_modes:
            raise ValueError(f"Streaming mode {mode} is not supported. Known modes are {', '.join(known_modes)}")
        
        self._stream_mode = mode
        
    @property
    def data_dir(self):
        return self._data_dir

    @data_dir.setter
    def data_dir(self, datadir):
        if not os.path.isdir(datadir):
            raise NotADirectoryError(f"Parsed data directory '{datadir}' does not exist.")

        self._data_dir = datadir

    @property
    def file_list(self):
        return self._file_list

    @file_list.setter
    def file_list(self, patt):
        patt = patt if patt.endswith(".nc") else f"{patt}.nc"
        files = glob.glob(os.path.join(self.data_dir, patt))

        if not files:
            raise FileNotFoundError(f"Could not find any files with pattern '{patt}' under '{self.data_dir}'.")

        self._file_list = list(
            np.asarray(sorted(files, key=lambda s: int(re.search(r'\d+', os.path.basename(s)).group()))))

    @property
    def seed(self):
        return self._seed 

    @seed.setter 
    def seed(self, seed_int):
        if self.with_horovod and seed_int is None:
            raise ValueError(f"Seed integer must be provided for distitributed training with Horovod,")
        
        # set seed
        random.seed(seed_int)
        self._seed = seed_int

    @property
    def nfiles2merge(self):
        return self._nfiles2merge
    
    @nfiles2merge.setter
    def nfiles2merge(self, n2merge: Union[int, Dict]):

        if isinstance(n2merge, int):
            n = n2merge
        else: 
            n = n2merge[f"#GPUS={hvd.size()}"] if self.with_horovod else n2merge[f"#GPUS=1"]

        # ensure that n is a divisor of the total number of files
        n = find_closest_divisor(self.nfiles, n)

        self._nfiles2merge = n
        # for distributed training, data files must be distributed over workers
        if self.with_horovod:
            if hvd.rank() == 0:
                if n != n2merge:
                    print(f"{n2merge} is not a divisor of the total number of files. Value is changed to {n}")
                print(f"Distributed streaming over {hvd.size()} workers.")

            assert n > hvd.size(), f"Number of files to merge {n} must be larger than number of workers {hvd.size()}."
            self._nfiles2merge = int(n / hvd.size())
            if n % hvd.size() > 0:
                # In case that the modulo is non-zero, nfiles2merge is incremented and the file list is appended
                # so that each work processes the same number of files.
                # Note that duplicated files only occur in the last data subset. 
                # To avoid duplicates in the last subset itself, only files from the preceiding subsets are appended.
                self._nfiles2merge += 1
                nfiles_req = int(hvd.size() * self._nfiles2merge * self.nfiles/n)
                if hvd.rank() == 0: 
                    print(f"Append file list by {nfiles_req - self.nfiles} files to get {nfiles_req} files ({self._nfiles2merge} files per worker).")
                self.file_list_random += random.sample(self.file_list_random[0:self.nfiles-n], nfiles_req - self.nfiles)
                self.nfiles = len(self.file_list_random)
        else:
            if n != n2merge:
                print(f"{n2merge} is not a divisor of the total number of files. Value is changed to {n}")


    @property
    def sample_dim(self):
        return self._sample_dim

    @sample_dim.setter
    def sample_dim(self, sample_dim):
        if not sample_dim in self.all_dims:
            raise KeyError(f"Could not find dimension '{sample_dim}' in data.")

        self._sample_dim = sample_dim

    @property
    def predictor_list(self):
        return self._predictor_list

    @predictor_list.setter
    def predictor_list(self, selected_predictors: List):
        """
        Initalizes predictor list. In case that selected_predictors is set to None, all variables with suffix `_in`
        in their names are selected.
        In case that a list of selected_predictors is parsed, their availability is checked
        :param selected_predictors: list of predictor variables or None
        """
        self._predictor_list = self.check_and_choose_vars(selected_predictors, "_in")
        
    @property
    def static_predictor_list(self):
        return self._static_predictor_list
    
    @static_predictor_list.setter
    def static_predictor_list(self, selected_static_predictors: List):
        if selected_static_predictors is None:
            # if no static, high-res predictors are added, set to None
            self._static_predictor_list = None
        else:
            self._static_predictor_list = self.check_and_choose_vars(selected_static_predictors)

    @property
    def predictand_list(self):
        return self._predictand_list

    @predictand_list.setter
    def predictand_list(self, selected_predictands: List):
        """
        Similar to predictor_list-setter, but does not allow for parsing None.
        """
        assert isinstance(selected_predictands, list), "Selected predictands must be a list of variable names"
        self._predictand_list = self.check_and_choose_vars(selected_predictands, "_tar")

    def __len__(self):
        return self.nsamples

    def getitems(self, indices):
        """
        Return samples from loaded dataset, either as single array (mode: 'hi_input' and 'hi_input_named_target')
        or as tuple of arrays (mode: 'lo_input')
        :param indices: sample indices 
        """
        if self.stream_mode == "lo_input":
            da_now = self.getitems_as_array(indices)
        else:
            da_now = self.getitems_as_tuple(indices)
        
        return da_now
    
    def getitems_as_array(self, indices):
        """
        Retrieves samples from dataset and returns an ordered array for later data handling.
        :param indices: sample indices 
        :return: Ordered array of variables with variables as last dimension. Order is: [static_predictors], predictors, predictands
        """
        da_now = self.data_now.isel({self.sample_dim: indices}).to_array("variables").sel({"variables": self.all_vars})
        
        return da_now.transpose(..., "variables")        
        
    def getitems_as_tuple(self, indices):
        """
        Retrieves samples from dataset and returns an ordered tuple of arrays for later data handling.
        :param indices: sample indices 
        :return: Ordered tuple of arrays with variables as last dimension. Order is: static_predictors, predictors, predictands
        """
        da_in_coa, da_in_static, da_out = self.data_now[self.predictor_list].isel({self.sample_dim: indices}).to_array("variables"), \
                                          self.data_now[self.static_predictor_list].isel({self.sample_dim: indices}).to_array("variables"), \
                                          self.data_now[self.predictand_list].isel({self.sample_dim: indices}).to_array("variables")
        
        da_tuple = (da_in_static.transpose(..., "variables"), da_in_coa.transpose(..., "variables"), da_out.transpose(..., "variables"))
        return da_tuple

    def get_dataset_size(self, random_list: bool = False):
        """
        Sum the size of all dataset files in bytes.
        :param random_list: if True, the size computation is based on randomized list which might be longer for distributed training
        :return: size of dataset files in bytes 
        """
        dataset_size = 0.
        # iterate over file_list_random-attribute since this comprises the actual files that are streamed 
        # incl. duplicates in case of distributed training
        flist = self.file_list_random if random_list else self.file_list 
        for datafile in flist:
            dataset_size += os.path.getsize(datafile)

        return dataset_size

    def get_nxy_dim(self, ds):
        """
        Retrieve the spatial dimensionality of the input and target data.
        :return: Dictionary of spatial dimensions of the predictands, predictors and, if available, static predictors
        """
        data_dims_keys = ["output", "input",]
        infer_vars = [self.predictand_list[0], self.predictor_list[0]]
        
        if self.static_predictor_list is not None:
            data_dims_keys += ["input_static"]
            infer_vars += [self.static_predictor_list[0]]

        dim_dict = {}
        for key, var in zip(data_dims_keys, infer_vars):
            dimnames = list(ds[var].dims)
            dimnames.remove(self.sample_dim)
            
            print(dimnames)
            print(self.all_dims)

            data_dim = itemgetter(*dimnames)(self.all_dims)
            dim_dict[key] = data_dim
            
        return dim_dict

    def get_samples_per_merged_file(self):
        nsamples_merged = []

        for i in range(self.nfiles_merged):
            file_list_now = self.file_list_random[i * self.nfiles2merge: (i + 1) * self.nfiles2merge]
            ds_now = xr.open_mfdataset(list(file_list_now), decode_cf=False)
            nsamples_merged.append(ds_now.dims[self.sample_dim])  

        return max(nsamples_merged)

    def get_all_varnames(self):
        ds_test = xr.open_dataset(self.file_list[0])
        return list(ds_test.variables)

    def check_and_choose_vars(self, var_list: List[str], suffix: str = "*"):
        """
        Checks list of variables for availability or retrieves all variables named with a given suffix
        (for var_list = None)
        :param var_list: list of predictor variables or None
        :param suffix: optional suffix of variables to selected. Only effective if var_list is None
        :return selected_vars: list of selected variables
        """
        if var_list is None:
            selected_vars = [var for var in self.varnames_list if var.endswith(suffix)]
        else:
            stat_list = [var in self.varnames_list for var in var_list]
            if all(stat_list):
                selected_vars = var_list
            else:
                miss_inds = [i for i, x in enumerate(stat_list) if not x]
                miss_vars = [var_list[i] for i in miss_inds]
                raise ValueError(f"Could not find the following variables in the dataset: {*miss_vars,}")

        return selected_vars
    
    def check_data_shapes(self):
        """
        Check if the spatial data dimensions are consistent w.r.t. to the streaming mode.
        """
        nxy_in_str, nxy_stat_str = [str(n) for n in self.data_xy_dim['input']], [str(n) for n in self.data_xy_dim['input_static']]
        nxy_out_str = [str(n) for n in self.data_xy_dim['output']]
        
        if self.stream_mode == "lo_input":
            assert self.data_xy_dim["input"] != self.data_xy_dim["input_static"], f"Predictors and static predictors must have different spatial shapes. " + \
                                                                                      f"predictors: [{','.join(nxy_in_str)}], " + \
                                                                                      f"static_predictors: [{','.join(nxy_stat_str)}]"
            assert self.data_xy_dim["output"] == self.data_xy_dim["input_static"], f"Predictands and static predictors must have the same spatial shapes. " + \
                                                                                      f"predictands: [{','.join(nxy_out_str)}], " + \
                                                                                      f"static_predictors: [{','.join(nxy_stat_str)}]"
        else:
            mess = f"The spatial shapes of all variables must be the same. predictands: [{','.join(nxy_out_str)}], predictors: [{','.join(nxy_in_str)}]"
            if self.static_predictor_list is not None:
                mess += f" static_predictors: [{','.join(nxy_stat_str)}]"
            assert self.data_xy_dim["input"] == self.data_xy_dim["input_static"] == self.data_xy_dim["output"], mess        

    @staticmethod
    def _process_one_netcdf(fname, data_norm, engine: str = "netcdf4", var_list: List = None, **kwargs):
        with xr.open_dataset(fname, decode_cf=False, engine=engine, **kwargs) as ds_now:
            if var_list: ds_now = ds_now[var_list]
            ds_now = StreamMonthlyNetCDF._preprocess_ds(ds_now, data_norm)
            ds_now = ds_now.load()
            return ds_now

    @staticmethod
    def _preprocess_ds(ds, data_norm):
        ds = data_norm.normalize(ds)
        return ds.astype("float32")

    def _read_mfdataset(self, files, **kwargs):
        # parallel processing of files incl. normalization
        datasets = self.pool.map(partial(self._process_one_netcdf, data_norm=self.data_norm, **kwargs), files)
        ds_all = xr.concat(datasets, dim=self.sample_dim)
        # clean-up
        del datasets
        gc.collect()

        return ds_all

    def read_netcdf(self, set_ind):
        set_ind = tf.keras.backend.get_value(set_ind)
        set_ind = int(str(set_ind).lstrip("b'").rstrip("'"))
        set_ind = int(set_ind % self.nfiles_merged)
        file_list_now = self.file_list_random[set_ind * self.nfiles2merge:(set_ind + 1) * self.nfiles2merge]
        il = int(self.iload_next % 2)
        # read the normalized data into memory
        # ds_now = xr.open_mfdataset(list(file_list_now), decode_cf=False, data_vars=self.all_vars,
        #                           preprocess=partial(self._preprocess_ds, data_norm=self.data_norm),
        #                           parallel=True).load()
        t0 = timer()
        # Restriction to read dynamic variables is not required currently,
        # since constant data get automatically broadcasted with the _read_mfdataset-method
        #data_now = self._read_mfdataset(file_list_now, var_list=self.dyn_vars).copy()
        data_now = self._read_mfdataset(file_list_now, var_list=self.all_vars).copy()
        nsamples = data_now.sizes[self.sample_dim]

        if nsamples < self.samples_merged:
            t1 = timer()
            add_samples = self.samples_merged - nsamples
            istart = random.randint(0, self.samples_merged - add_samples - 1)
            # slice data from data_now...
            ds_add = data_now.isel({self.sample_dim: slice(istart, istart+add_samples)})
            if ds_add.sizes[self.sample_dim] != add_samples:
                print("WARNING: ds_add contains inconsistent number of samples. Re-try...")
                add_samples = self.samples_merged - nsamples
                istart = random.randint(0, self.samples_merged - add_samples - 1)
                ds_add = data_now.isel({self.sample_dim: slice(istart, istart + add_samples)})
            # ... and modify underlying sample-dimension to allow clean concatenation
            ds_add[self.sample_dim] = data_now[self.sample_dim][-1].values + 1 + np.arange(add_samples)
            ds_add[self.sample_dim] = ds_add[self.sample_dim].assign_attrs(data_now[self.sample_dim].attrs)
            data_now = xr.concat([data_now, ds_add], dim=self.sample_dim)
            print(f"Appending data with {add_samples:d} samples took {timer() - t1:.2f}s" +
                  f"(total #samples: {data_now.sizes[self.sample_dim]})")
            
        # Appending with constant variables is not required since they are read and broadcast to data_now already (see above)
        #if self.const_vars:
        #    ds_const_append = self.ds_const.copy().expand_dims({self.sample_dim: data_now[self.sample_dim]})
        #    data_now = xr.merge([data_now, ds_const_append])

        # write to class attribute
        self.data_loaded[il] = data_now
        # timing
        t_read = timer() - t0
        self.reading_times.append(t_read)
        self.ds_proc_size += data_now.nbytes
        print(f"Dataset #{set_ind:d} ({il+1:d}/2) reading time: {t_read:.2f}s.")
        self.iload_next = il + 1

        return il

    def choose_data(self, _):
        ik = int(self.iuse_next % 2)
        self.data_now = self.data_loaded[ik]
        print(f"Use data subset {ik:d}...")
        self.iuse_next = ik + 1
        return True

# Modified normalization class
Note that this is mandatory since the data has differing coordinates for target and input data is not yet supported by the normalization class.
To-Do:
- [ ] Revise Normalize-class accordingly
Here, an ad-hoc fix is made to allow passing of `norm_dims=None` which results into averaging over all data dimensions.

In [ ]:
da_or_ds = Union[xr.DataArray, xr.Dataset]

class Normalize(ABC):
    """
    Abstract class for normalizing data.
    """

    def __init__(self, method: str, norm_dims: List):
        self.method = method
        self.norm_dims = norm_dims
        self.norm_stats = None

    def normalize(self, data: xr.DataArray, **stats):
        """
        Normalize data
        :param data: The DataArray to be normalized
        :param stats: Optional parameters to perform normalization. Must fit to normalization type!
        :return: DataArray with normalized data
        """
        # sanity checks
        # if not isinstance(data, xr.DataArray):
        #    raise TypeError(f"Passed data must be a xarray.DataArray, but is of type {str(type(data))}.")

        # do the computation
        norm_stats = self.get_required_stats(data, **stats)
        norm_stats = Normalize.match_datatype(data, *norm_stats)
        data_norm = self.normalize_data(data, *norm_stats)

        return data_norm

    def denormalize(self, data: da_or_ds, **stats):
        """
        Denormalize data.
        :param data: The DataArray to be denormalized.
        :param stats: Optional parameters to perform denormalization. Must fit to normalization type!
        :return: DataArray with denormalized data.
        """
        # sanity checks
        # if not isinstance(data, xr.DataArray):
        #    raise TypeError(f"Passed data must be a xarray.DataArray, but is of type {str(type(data))}.")

        # do the computation
        norm_stats = self.get_required_stats(data, **stats)
        norm_stats = Normalize.match_datatype(data, *norm_stats)
        data_denorm = self.denormalize_data(data, *norm_stats)

        return data_denorm

    @property
    def norm_dims(self):
        return self._norm_dims

    @norm_dims.setter
    def norm_dims(self, norm_dims):
        self._norm_dims = list(norm_dims) if norm_dims is not None else None

    def _check_norm_dims(self, data):
        """
        Check if dimension for normalization reside in dimensions of data.
        :param data: the data (xr.DataArray) to be normalized
        :return True: in case of passed check, a ValueError is risen else
        """
        data_dims = list(data.dims)
        norm_dims_check = [norm_dim in data_dims for norm_dim in self.norm_dims]
        if not all(norm_dims_check):
            imiss = np.where(~np.array(norm_dims_check))[0]
            miss_dims = list(np.array(self.norm_dims)[imiss])
            raise ValueError("The following dimensions do not reside in the data: " +
                             f"{', '.join(miss_dims)}")

        return True

    @staticmethod
    def match_datatype(data, *args, var_dim="variables"):
        """
        Ensures that the arguments have the same xarray datatype (either xr.DataArray or xr.Dataset) as data,
        i.e. coerces all arguments against type(data) if necessary.
        :param data: the reference data (must be either xr.Dataset or xr.DataArray)
        :param args: arbitrary number of arguments (all of them must also be either xr.Dataset or xr.DataArray,
                     but should not be mixed, e.g. type(args[0])=xr.Dataset and type(args[1])=xr.DataArray is not
                     allowed
        :param var_dim: dimension name to convert from/to xr.Dataset/xr.DataArray
        """

        # sanity check
        ds_or_da = (xr.Dataset, xr.DataArray)
        all_args = [data] + list(args)
        if not all(isinstance(arg, ds_or_da) for arg in all_args):
            flags = [not isinstance(arg, ds_or_da) for arg in all_args]
            inds = np.nonzero(flags)[0].tolist()
            if len(inds) == 1:
                err_str = f"The parsed argument at position {inds} is"
            else:
                err_str = f"The parsed arguments at positions {inds} are"
            raise ValueError(f"{err_str} not an xarray.DataArray or xarray.Dataset.")

        # align type of arguments if required
        if isinstance(data, type(args[0])):
            args_new = args
        elif isinstance(data, xr.Dataset) and isinstance(args[0], xr.DataArray):
            args_new = tuple(arg.to_dataset(dim=var_dim) for arg in args)
        elif isinstance(data, xr.DataArray) and isinstance(args[0], xr.Dataset):
            args_new = tuple(arg.to_array(dim=var_dim) for arg in args)
        else:
            raise ValueError("Unknown error occured. Please check all input parameters.")

        return args_new

    def save_norm_to_file(self, js_file, missdir_ok: bool = True):
        """
        Write normalization parameters to file
        :param js_file: Path to JSON-file to be created
        :param missdir_ok: If True, base-directory of JSON-file can be missing and will be created then
        :return: -
        """
        if self.norm_stats is None:
            raise AttributeError("norm_stats is still None. Please run (de-)normalization to get parameters.")

        if any([stat is None for stat in self.norm_stats.values()]):
            raise AttributeError("Some parameters of norm_stats are None.")

        norm_serialized = {key: da.to_dict() for key, da in self.norm_stats.items()}

        # serialization and (later) deserialization depends on data type.
        # Thus, we have to save it to the dictionary
        d0 = list(self.norm_stats.values())[0]
        if isinstance(d0, xr.DataArray):
            norm_serialized["data_type"] = "data_array"
        elif isinstance(d0, xr.Dataset):
            norm_serialized["data_type"] = "data_set"

        if missdir_ok: os.makedirs(os.path.dirname(js_file), exist_ok=True)

        with open(js_file, "w") as jsf:
            js.dump(norm_serialized, jsf)

    def read_norm_from_file(self, js_file):
        """
        Read normalization parameters from file. Inverse function to write_norm_from_file.
        :param js_file: Path to JSON-file to be read.
        :return: Parameters set to self.norm_stats
        """
        with open(js_file, "r") as jsf:
            norm_data = js.load(jsf)

        data_type = norm_data.pop('data_type', None)

        if data_type == "data_array":
            xr_obj = xr.DataArray
        elif data_type == "data_set":
            xr_obj = xr.Dataset
        else:
            raise ValueError(
                f"Unknown data_type {data_type} in {js_file}. Only 'data_array' or 'data_set' are allowed.")

        norm_data.pop('data_type', None)

        norm_dict_restored = {key: xr_obj.from_dict(da_dict) for key, da_dict in norm_data.items()}

        self.norm_stats = norm_dict_restored

    @abstractmethod
    def get_required_stats(self, data, varname, *stats):
        """
        Function to retrieve either normalization parameters from data or from keyword arguments
        """
        pass

    @staticmethod
    @abstractmethod
    def normalize_data(data, *norm_param):
        """
        Function to normalize data.
        """
        pass

    @staticmethod
    @abstractmethod
    def denormalize_data(data, *norm_param):
        """
        Function to denormalize data.
        """
        pass

da_or_ds = Union[xr.DataArray, xr.Dataset]


class ZScore(Normalize):
    def __init__(self, norm_dims: List):
        super().__init__("z_score", norm_dims)
        self.norm_stats = {"mu": None, "sigma": None}

    def get_required_stats(self, data: da_or_ds, varname: str= None, **stats):
        """
        Get required parameters for z-score normalization. They are either computed from the data
        or can be parsed as keyword arguments.
        :param data: the data to be (de-)normalized
        :param varname: retrieve parameters for specific varname only (without effect if parameters must be retrieved from data)
        :param stats: keyword arguments for mean (mu) and standard deviation (std) used for normalization
        :return (mu, sigma): Parameters for normalization
        """
        mu, std = stats.get("mu", self.norm_stats["mu"]), stats.get("sigma", self.norm_stats["sigma"])

        if mu is None or std is None:
            print("Retrieve mu and sigma from data...")
            mu, std = data.mean(self.norm_dims), data.std(self.norm_dims)
            # the following ensure that both parameters are computed in one graph!
            # This significantly reduces memory footprint as we don't end up having data duplicates
            # in memory due to multiple graphs (and also seem to enfore usage of data chunks as well)
            mu, std = dask.compute(mu, std)
            self.norm_stats = {"mu": mu, "sigma": std}
        else:
            if varname:
                if isinstance(mu, xr.DataArray):
                    mu, std = mu.sel({"variables": varname}), std.sel({"variables": varname})
                elif isinstance(mu, xr.Dataset):
                    mu, std = mu[varname], std[varname]
                else:
                    raise ValueError(f"Unexpected data type for mu and std: {type(mu)}, {type(std)}")
        #    print("Mu and sigma are parsed for (de-)normalization.")

        return mu, std

    @staticmethod
    def normalize_data(data, mu, std):
        """
        Perform z-score normalization on data
        :param data: Data array of interest
        :param mu: mean of data for normalization
        :param std: standard deviation of data for normalization
        :return data_norm: normalized data
        """
        data = (data - mu) / std

        return data

    @staticmethod
    def denormalize_data(data, mu, std):
        """
        Perform z-score denormalization on data.
        :param data: Data array of interest
        :param mu: mean of data for denormalization
        :param std: standard deviation of data for denormalization
        :return data_norm: denormalized data
        """
        data = data * std + mu

        return data

## Test make_tf_dataset_allmem data pipeline (streaming from a single netCDF)

### Get the data
Stream data from example data-file.

In [ ]:
# set diretcories and (hyper-)parameters for WGAN
data_dir = Path("/p/scratch/deepacf/maelstrom/maelstrom_data/ap5/downscaling_benchmark_dataset/benchmark_t2m/dataset/coarse_input/val")
t2m_test_file = data_dir.joinpath("downscaling_benchmark_t2m_val.nc")
js_norm = data_dir.parents[0].joinpath("norm.json")
# datadir = "/p/scratch/deepacf/maelstrom/maelstrom_data/ap5_michael/preprocessed_era5_ifs/netcdf_data/all_files/"
# outdir = "/p/project/deepacf/maelstrom/langguth1/downscaling_jsc_repo/downscaling_unet/trained_models"

lr_gen = 5.0e-05
lr_gen_end = lr_gen / 10.0
lr_critic = 1.0e-06
lr_decay = True
nepochs = 1
d_steps = 5
batch_size_demo = 2

# get normalization instance
data_norm = ZScore(None)
data_norm.read_norm_from_file(js_norm)

# read raw data...
ds_train = xr.open_dataset(t2m_test_file)

#...and normalize
ds_train = data_norm.normalize(ds_train)
ds_val = ds_train
z_branch = False
print("Datasets for trining, validation and testing loaded.")

# wgan_model = HarrisWGAN(GeneratorHarris, DiscriminatorHarris,
#                  {"lr_decay": lr_decay, "lr_gen": lr_gen, "lr_critic": lr_critic, "lr_gen_end": lr_gen_end,
#                   "train_epochs": nepochs, "d_steps": d_steps, "z_branch": z_branch})

Set-up data pipeline (same for training and validation for simplicity)

In [ ]:
tfds = make_tf_dataset_allmem(
    ds_train,
    batch_size_demo * (d_steps + 1),
    ["t_2m_tar"],
    ["t2m_in", "sp_in", "sshf_in", "t_ml115_in"],
    ["fr_land_tar", "hsurf_tar"],
)
tfds_val = make_tf_dataset_allmem(
    ds_train,
    batch_size_demo * (d_steps + 1),
    ["t_2m_tar"],
    ["t2m_in", "sp_in", "sshf_in", "t_ml115_in"],
    ["fr_land_tar", "hsurf_tar"],
)

In [ ]:
tfds.element_spec[0]

### Start training 

In [ ]:
importlib.reload(sys.modules["model_engine"])
importlib.reload(sys.modules["harris_wgan_model"])

# some prerequisites to instantiate the model and run training
shape_in = {
    "harris_generator": {
        "lo_res_inputs": (32, 36, 4),
        "hi_res_inputs": (128, 144, 2),
        "noise_input": (32, 36, 4),
    },
    "harris_discriminator": {
        "lo_res_inputs": (32, 36, 4),
        "hi_res_inputs": (128, 144, 2),
        "output": (128, 144, 1),
    },
}
varnames_tar = "t_2m_tar"
hparams_dict = dict()                    # make use of defaults!
model_savedir = ""
steps_per_epoch = 10

model_instance = ModelEngine("harris_wgan")
# data prep here
model = model_instance(
    shape_in, list(varnames_tar), hparams_dict, model_savedir, "demo"
)
model.compile(**model.compile_options)
history = model.fit(
    x=tfds,
    epochs=model.hparams["nepochs"],
    steps_per_epoch=steps_per_epoch,
    validation_data=tfds_val,
    validation_steps=steps_per_epoch,
    verbose=1,
    **model.fit_options
)

In [ ]:
model_name = "harriswgan_lr1e-05_epochs1_opt_split_era5_ifs"

savedir = os.path.join("../downscaling_harriswgan/trained_models/", model_name)
os.makedirs(savedir, exist_ok=True)

In [ ]:
model.generator.save(
    os.path.join(savedir, "harriswgan_lr1e-05_epochs30_demo_generator")
)
model.critic.save(
    os.path.join(savedir, "harriswgan_lr1e-05_epochs30_demo_discriminator")
)

## Test make_tf_dataset_dyn data pipeline (streaming from a set of netCDF-file)

In [ ]:
# paramters
datadir = "/p/scratch/deepacf/maelstrom/maelstrom_data/ap5/downscaling_benchmark_dataset/benchmark_t2m/dataset/coarse_input/val"
file_patt = "downscaling_benchmark_t2m_train_*.nc"
stream_mode = "lo_input"
nfiles_load = 4
norm_dims = ["rlat_in", "rlon_in", "rlat_tar", "rlon_tar"]
nworkers = 4
batch_size = 2
nepochs = 1
    
predictands = ["t_2m_tar"]
predictors= ["t2m_in", "slhf_in", "sshf_in", "sp_in", "z_in", "lsm_in", "t_ml115_in", "t_ml122_in", "t_ml127_in", "t_ml131_in", "t_ml135_in"]
static_predictors= ["hsurf_tar", "fr_land_tar"]


### Get the data
Stream data from example data-file.

In [ ]:
ds_obj = StreamMonthlyNetCDF(stream_mode, datadir, file_patt, nfiles_load, predictands, predictors, static_predictors, 
                             norm_dims = norm_dims, nworkers=nworkers)

tfds=make_tf_dataset_dyn(ds_obj, batch_size=batch_size, nepochs=nepochs, nshuffle = 1000, lrepeat= True, drop_remainder = True)

In [ ]:
tfds.element_spec[0]

In [ ]:
ds_obj.data_xy_dim["input_static"] = (128, 144)
  
print(ds_obj.data_xy_dim["input"])
print(ds_obj.data_xy_dim["input_static"])
print(ds_obj.data_xy_dim["output"])

if ds_obj.stream_mode == "lo_input":
    nxy_in_str, nxy_stat_str, nxy_out_str = [str(n) for n in ds_obj.data_xy_dim['input']], [str(n) for n in ds_obj.data_xy_dim['input_static']], [str(n) for n in ds_obj.data_xy_dim['output']]
    assert ds_obj.data_xy_dim["input"] != ds_obj.data_xy_dim["input_static"], f"Predictors and static predictors must have different spatial shapes. " + \
                                                                              f"predictors: [{','.join(nxy_in_str)}], " + \
                                                                              f"static_predictors: [{','.join(nxy_stat_str)}]"
    assert ds_obj.data_xy_dim["output"] == ds_obj.data_xy_dim["input_static"], f"Predictands and static predictors must have the same spatial shapes. " + \
                                                                              f"predictands: [{','.join(nxy_out_str)}], " + \
                                                                              f"static_predictors: [{','.join(nxy_stat_str)}]"
else:
    assert ds_obj.data_xy_dim["input"] == ds_obj.data_xy_dim["input_static"] == ds_obj.data_xy_dim["output"]

In [ ]:
if ds_obj.stream_mode == "lo_input":
    shape_in = [*ds_obj.data_xy_dim["input"], len(ds_obj.predictor_list), len(ds_obj.static_predictor_list)]
else:
    shape_in = [*ds_obj.data_xy_dim["input"], len(ds_obj.predictor_list + ds_obj.static_predictor_list)]

In [ ]:
tuple(shape_in)

In [ ]:
dimnames = ['rlat_tar', 'rlon_tar']
all_dims = ['time', 'rlon_in', 'rlat_in', 'rlon_tar', 'rlat_tar']

data_dim = itemgetter(*dimnames)(all_dims)

In [ ]:
ds_all.dims[sample_dim]

# Correcting inference for HarrisWGAN

### Problem statement
Current checkpointed model cnanot e evaluated with main_postprocessing.py since the checkpointed model requires the data pipeline to provide a `noise_input`.
Subsequently, it will be tested if this error can be circumvented. In particular, attempts will be made to modify the `predict_step`-method of the generator (not the Harris WGAN).

In [62]:
# %load /p/home/jusers/langguth1/juwels/downscaling_maelstrom/downscaling_benchmark/models/harris_wgan_model.py
# Harris et al 2022, WGAN model implementation
"""
Class for Harris et al 2022, conditional Wasserstein GAN model (CWGAN)
"""
import os, sys
sys.path.append("../models/")
sys.path.append("../utils/")
sys.path.append("../handle_data/")

from pathlib import Path
from typing import List, Tuple, Union, Dict
from collections import OrderedDict
import types
import glob
import pickle
import json as js
import numpy as np
from abstract_model_class import AbstractModelClass
import tensorflow as tf
import tensorflow.keras as keras
from tensorflow.python.keras.utils import tf_utils
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from tensorflow.keras.layers import Input, concatenate, LeakyReLU, UpSampling2D, Layer, BatchNormalization, Conv2D, Add, AveragePooling2D, GlobalAveragePooling2D, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.utils import plot_model as k_plot_model
from tensorflow.keras import backend as K
from model_engine import ModelEngine
from custom_losses import get_custom_loss
from wgan_model import LearningRateSchedulerWGAN
from tensorflow.python.platform import tf_logging as logging
from handle_data_class import prepare_dataset
from all_normalizations import ZScore
from abstract_data_normalization import Normalize


list_or_tuple = Union[List, Tuple]

class GeneratorHarris(AbstractModelClass):
    # content based on original generator function from models.py (Harris repo)
    # structure based on Critic_Simple from wgan_model.py
    def __init__(self, shape_in: List, hparams: dict, varnames_tar: List, hparams_wgan):
        # Pass empty savedir- and expname-arguments since this is not a stand-alone model
        super().__init__(shape_in, hparams, varnames_tar, "", "")
        
        # set submodels
        self.set_hparams(hparams)
        self.hparams_wgan = hparams_wgan
        self.set_model()
        
        self.noise_gen = NoiseGenerator(
            self._input_shape["lo_res_inputs"][:2]+[self.hparams_wgan["noise_channels"]],
            self.hparams_wgan["batch_size"]*(self.hparams_wgan["d_steps"] + 1)
        )
        
    def set_model(self):        
        ds_steps = self.hparams["ds_steps"]
        filters_gen = self.hparams["channels_start"]
        kernel = self.hparams["kernel"]
        relu_alpha = self.hparams["relu_alpha"]
        padding = self.hparams["padding"]
        # Network inputs
        # low resolution condition
        generator_input = Input(shape=self._input_shape["lo_res_inputs"], name="lo_res_inputs")
        # constant fields
        const_input = Input(shape=self._input_shape["hi_res_inputs"], name="hi_res_inputs")

        # Convolve constant fields down to match other input dimensions
        upscaled_const_input = const_upscale_block(
            const_input, steps=ds_steps, filters=filters_gen
        )
        # noise
        noise_input = Input(shape=self._input_shape["noise_input"], name="noise_input")
        # Concatenate all inputs together
        generator_output = concatenate(
            [generator_input, upscaled_const_input, noise_input]
        )

        # Pass through 3 residual blocks
        for ii in range(3):
            generator_output = residual_block(
                generator_output,
                filters=filters_gen,
                conv_size=kernel,
                stride=1,
                relu_alpha=relu_alpha,
                padding=padding,
            )

        # Upsampling from low-res to high-res with alternating residual blocks
        # In the paper, this was [2*filters_gen, filters_gen] for steps of 5 and 2
        block_channels = [2 * filters_gen] * (len(ds_steps) - 1) + [filters_gen]
        for ii, step in enumerate(ds_steps):
            generator_output = UpSampling2D(size=(step, step), interpolation="bilinear")(
                generator_output
            )

            generator_output = residual_block(
                generator_output,
                filters=block_channels[ii],
                conv_size=kernel,
                stride=1,
                relu_alpha=relu_alpha,
                padding=padding,
            )

        # Concatenate with original size constants field
        generator_output = concatenate([generator_output, const_input])

        # Pass through 3 residual blocks
        for ii in range(3):
            generator_output = residual_block(
                generator_output,
                filters=filters_gen,
                conv_size=kernel,
                stride=1,
                relu_alpha=relu_alpha,
                padding=padding,
            )

        # Output layer
        actv_out = "linear"
        generator_output = Conv2D(
            filters=1, kernel_size=(1, 1), activation=actv_out, name="output"
        )(generator_output)
        
        self.model = Model(
            inputs=[generator_input, const_input, noise_input],
            outputs=generator_output,
            name="gen", 
        )
        
                   
    def set_compile_options(self):
        raise RuntimeError(f"Generator model is supposed to be part of a composite model such as WGAN, but not as standalone model for training.")
        
    def set_fit_options(self):
        raise RuntimeError(f"Generator model is supposed to be part of a composite model such as WGAN, but not as standalone model for training.")
        
    def set_hparams_default(self):
        """
        Note: hyperparameter defaults of generator and discriminator model must be set in the respective model classes whose instances are just parsed here.
        """
        self.hparams_default = {"channels_start": 128, "activation": "leaky_relu", "kernel": (3, 3), "stride": (2, 2), "lr": 1.e-5, 
                                "ds_steps": [4,], "padding": "reflect", "relu_alpha": 0.2, "lr_end": 1.e-05}


class DiscriminatorHarris(AbstractModelClass):
    # content based on original discriminator function from models.py (Harris repo)
    # structure based on Critic_Simple from wgan_model.py
    def __init__(self, shape_in: List, hparams: dict, varnames_tar: List):
        # Pass empty savedir- and expname-arguments since this is not a stand-alone model
        super().__init__(shape_in, hparams, varnames_tar, "", "")
        
        # set submodels
        self.set_hparams(hparams)
        self.set_model()
        
    def set_model(self):
        ds_steps = self.hparams["ds_steps"]
        filters_disc = self.hparams["channels_start"]
        kernel = self.hparams["kernel"]
        relu_alpha = self.hparams["relu_alpha"]
        padding = self.hparams["padding"]
        # Network inputs
        # low resolution condition
        generator_input = Input(shape=self._input_shape["lo_res_inputs"], name="lo_res_inputs")
        # constant fields
        const_input = Input(shape=self._input_shape["hi_res_inputs"], name="hi_res_inputs")
        # target image
        generator_output = Input(shape=self._input_shape["output"], name="output")

        # convolve down constant fields to match ERA
        lo_res_const_input = const_upscale_block(
            const_input, steps=ds_steps, filters=filters_disc
        )

        # concatenate constants to lo-res input
        lo_res_input = concatenate([generator_input, lo_res_const_input])

        # concatenate constants to hi-res input
        hi_res_input = concatenate([generator_output, const_input])

        # encode inputs using residual blocks
        # In the paper, this was [filters_disc, 2*filters_disc] for steps of 5 and 2
        block_channels = [filters_disc] * (len(ds_steps) - 1) + [2 * filters_disc]

        for ii, step in enumerate(ds_steps):
            lo_res_input = residual_block(
                lo_res_input,
                filters=block_channels[ii],
                conv_size=kernel,
                stride=1,
                relu_alpha=relu_alpha,
                padding=padding,
            )
            hi_res_input = Conv2D(
                filters=block_channels[ii],
                kernel_size=(step, step),
                strides=step,
                padding="valid",
                activation="relu",
            )(hi_res_input)

            hi_res_input = residual_block(
                hi_res_input,
                filters=block_channels[ii],
                conv_size=kernel,
                stride=1,
                relu_alpha=relu_alpha,
                padding=padding,
            )

        # concatenate hi- and lo-res inputs channel-wise before passing through discriminator
        disc_input = concatenate([lo_res_input, hi_res_input])

        # encode in residual blocks
        disc_input = residual_block(
            disc_input,
            filters=filters_disc,
            conv_size=kernel,
            stride=1,
            relu_alpha=relu_alpha,
            padding=padding,
        )

        # discriminator output
        disc_output = GlobalAveragePooling2D()(disc_input)
        disc_output = Dense(64, activation="relu")(disc_output)
        disc_output = Dense(1, name="disc_output")(disc_output)

        self.model = Model(
            inputs=[generator_input, const_input, generator_output],
            outputs=disc_output,
            name="disc",
        )
                   
    def set_compile_options(self):
        raise RuntimeError(f"discriminator model is supposed to be part of a composite model such as WGAN, but not as standalone model for training.")
        
    def set_fit_options(self):
        raise RuntimeError(f"discriminator model is supposed to be part of a composite model such as WGAN, but not as standalone model for training.")
        
    def set_hparams_default(self):
        """
        Note: hyperparameter defaults of generator and discriminator model must be set in the respective model classes whose instances are just parsed here.
        """
        self.hparams_default = {"channels_start": 512, "activation": "leaky_relu", "kernel": (3, 3), "stride": (2, 2), 
                                "lr": 1.e-5, "ds_steps": [4,], "padding": "reflect", "relu_alpha": 0.2, "lr_end": 1.e-06}

    
class NoiseGenerator(object):
    """Used for the Generator to generate the random noise input"""
    def __init__(self, noise_shapes, batch_size=32, random_seed=None):
        self.noise_shapes = noise_shapes
        self.batch_size = batch_size
        self.prng = np.random.RandomState(seed=random_seed)

    def noise(self, shape, mean, std, batch_size: int = None):
        bs = self.batch_size if batch_size is None else batch_size 
        
        shape = [bs] + shape
        n = self.prng.randn(*shape).astype(np.float32)
        # n = np.zeros(shape, dtype=np.float32)
        if std != 1.0:
            n *= std
        if mean != 0.0:
            n += mean
        return n
    
    def __call__(self, mean=0.0, std=1.0):
        return self.noise(self.noise_shapes, mean, std)
    
    

class HarrisWGAN_Model(keras.Model):
    def __init__(self, generator, discriminator, hparams, noise_gen=None):
        super().__init__()
        self.generator = generator
        self.discriminator = discriminator
        self.hparams = hparams 
        self.noise_gen = noise_gen
        if noise_gen:
            assert isinstance(noise_gen, NoiseGenerator), "Parsed noise_gen must be an instance of the NoiseGenerator-class"
        
    def compile(self, optimizer, loss, **kwargs):
        super().compile(**kwargs)
        self.c_optimizer, self.g_optimizer = optimizer
        
        if not self.noise_gen:
            self.noise_gen = NoiseGenerator(
                self.generator._input_shape["lo_res_inputs"][:2]+[self.hparams["noise_channels"]],
                self.hparams["batch_size"]*(self.hparams["d_steps"] + 1)
            )
        
        # losses
        self.discriminator_loss = self.discriminator_loss #get_custom_loss("critic")
        self.discriminator_gen_loss = self.generator_loss #get_custom_loss("critic_generator")
        self.recon_loss = CL_chooser(self.hparams["recon_loss"])
        
    @tf.function    
    def train_step(self, data_iter: Dict, embed=None) -> OrderedDict:
        inputs, outputs = data_iter
        cond = inputs["lo_res_inputs"]
        const = inputs["hi_res_inputs"]
        if self.hparams["ensemble_size"] is None:
            noise = self.noise_gen()
        else:
            # ensemble stacked in an additional dimension at the end
            noise = tf.stack([self.noise_gen() for _ in range(self.hparams["ensemble_size"] + 1)], axis=-1)
        sample = outputs["output"]

        # train discriminator
        for i in range(self.hparams["d_steps"]):
            with tf.GradientTape() as tape_critic:
                
                ist, ie = i * self.hparams["batch_size"], (i + 1) * self.hparams["batch_size"]
                cond_iter = cond[ist:ie, ...]
                const_iter = const[ist:ie, ...]
                sample_iter = sample[ist:ie, ...]
                noise_iter = noise[ist:ie, ..., 0] # only take the first ensemble member

                gen_in = [cond_iter] + [const_iter] + [noise_iter]
                gen_out = self.generator.model(gen_in, training=True)
                disc_in_gen = [cond_iter] + [const_iter] + [gen_out]
                disc_in_gt = [cond_iter] + [const_iter] + [sample_iter]
                
                # calculate discriminators for both, the real and the generated data
                discriminator_gen = self.discriminator.model(disc_in_gen, training=True)
                discriminator_gt = self.discriminator.model(disc_in_gt, training=True)
                # calculate the loss (incl. gradient penalty)
                c_loss = self.discriminator_loss(discriminator_gt, discriminator_gen)
                #gp = GradientPenalty()([sample_iter, gen_out])
                gp = self.gradient_penalty(sample_iter, gen_out, cond_iter, const_iter)
                d_loss = c_loss + self.hparams["gp_weight"] * gp

            # calculate gradients and update discrimintor
            d_gradient = tape_critic.gradient(d_loss, self.discriminator.trainable_variables)
            self.c_optimizer.apply_gradients(zip(d_gradient, self.discriminator.trainable_variables))

        # train generator
        with tf.GradientTape() as tape_generator:
            # generate (downscaled) data
            cond_iter = cond[-self.hparams["batch_size"]:, ...]
            const_iter = const[-self.hparams["batch_size"]:, ...]
            noise_iter = noise[-self.hparams["batch_size"]:, ...]
            sample_iter = sample[-self.hparams["batch_size"]:, ...]

            # train generator for each ensemble member
            noise_iter_k = noise_iter[..., 0]
            gen_in = [cond_iter] + [const_iter] + [noise_iter_k]
            gen_data = self.generator.model(gen_in, training=True)
            gen_data_list = [gen_data]
            if self.hparams["ensemble_size"] is not None:
                gen_iter_list = []
                for k in range(self.hparams["ensemble_size"]):
                    noise_iter_k = noise_iter[..., k+1]
                    gen_in = [cond_iter] + [const_iter] + [noise_iter_k]
                    gen_data_iter = self.generator.model(gen_in, training=True)
                    gen_iter_list.append(gen_data_iter)
                    
                gen_data_list.append(tf.stack(gen_iter_list))
            
            disc_in_gen = [cond_iter] + [const_iter] + [gen_data]
            discriminator_gen = self.discriminator.model(disc_in_gen, training=True)

            # critic loss for generator
            cg_loss = self.discriminator_gen_loss(discriminator_gen)
            # content loss term
            cl_loss = self.recon_loss(sample_iter, gen_data_list[-1])
            # combined loss for generator
            g_loss = cg_loss + cl_loss*self.hparams["recon_weight"]


        g_gradient = tape_generator.gradient(g_loss, self.generator.trainable_variables)
        self.g_optimizer.apply_gradients(zip(g_gradient, self.generator.trainable_variables))

        return OrderedDict(
            [
                ("c_loss", c_loss),
                ("gp_loss", self.hparams["gp_weight"] * gp),
                ("d_loss", d_loss),
                ("cg_loss", cg_loss),
                #("recon_loss", cl_loss),
                ("recon_loss", cl_loss * self.hparams["recon_weight"]),
                ("g_loss", g_loss)
            ]
        )
            

    def test_step(self, val_iter: tf.data.Dataset) -> OrderedDict:
        """
        Implement step to test trained generator on validation data
        :param val_iter: Tensorflow Dataset with validation data
        :return: dictionary with reconstruction loss on validation data
        
        NOTE SL: taken from wgan_model.py
        """
        inputs, outputs = val_iter
        cond = inputs["lo_res_inputs"]
        const = inputs["hi_res_inputs"]
        sample = outputs["output"]
        if self.hparams["ensemble_size"] is None:
            noise = self.noise_gen()
        else:
            # ensemble stacked in an additional dimension at the end
            noise = tf.stack([self.noise_gen() for _ in range(self.hparams["ensemble_size"] + 1)], axis=-1)
        
        noise_iter = noise[0:self.hparams["batch_size"]:, ...]
        noise_0 = noise_iter[..., 0]
        gen_in = [cond] + [const] + [noise_0]
        gen_data = self.generator.model(gen_in, training=True)
        gen_data_list = [gen_data]
        if self.hparams["ensemble_size"] is not None:
            gen_iter_list = []
            for k in range(self.hparams["ensemble_size"]):
                noise_k = noise_iter[..., k+1]
                gen_in = [cond] + [const] + [noise_k]
                gen_data_k = self.generator.model(gen_in, training=True)
                gen_iter_list.append(gen_data_k)

            gen_data_list.append(tf.stack(gen_iter_list))

        disc_in_gen = [cond] + [const] + [gen_data]
        discriminator_gen = self.discriminator.model(disc_in_gen, training=True)

        # critic loss for generator
        cg_loss = self.discriminator_gen_loss(discriminator_gen)
        # content loss term
        cl_loss = self.recon_loss(sample, gen_data_list[-1])

        return OrderedDict([
            ("cg_loss", cg_loss),
            ("recon_loss", cl_loss * self.hparams["recon_weight"]),
        ])

    def predict_step(self, test_iter: tf.data.Dataset) -> OrderedDict:
        inputs, _ = test_iter
        cond = inputs["lo_res_inputs"]
        const = inputs["hi_res_inputs"]
        
        if self.hparams["ensemble_size"] is not None:
            noise = [self.noise_gen() for _ in range(self.hparams["ensemble_size"])]
            gen_list = []
            for noise_iter in noise:
                gen_in = [cond] + [const] + [noise_iter]
                gen_iter = self.generator(gen_in, training=False)
                gen_list.append(gen_iter)
            gen_out = tf.stack(gen_list, axis=-1)
        else:
            noise = self.noise_gen()
            gen_in = [cond] + [const] + [noise]
            gen_out = self.generator(gen_in, training=False)
        return gen_out

    def gradient_penalty(self, real_data, gen_data, cond_data, const_data):
        """
        Calculates gradient penalty based on 'mixture' of generated and ground truth data
        :param real_data: the ground truth (high-res) data
        :param gen_data: the generated/predicted (high-res) data
        :param cond_data: the conditional (low-res) input data of the generator/critic
        :param const_data: the static (high-res) data of the generator/critic
        :return: gradient penalty
        
        NOTE ML: This is now equivalent to the sage of the GradientPenalty-layer in the original WGAN-implementation,
                 cf. https://github.com/ECMWFCode4Earth/tesserugged/blob/561733660f53a2d3beedc55b593ba68ec260040e/dev/gan/dsrnngan/gan.py#L129C67-L129C69
                 and https://github.com/ECMWFCode4Earth/tesserugged/blob/561733660f53a2d3beedc55b593ba68ec260040e/dev/gan/dsrnngan/layers.py#L13
        
        """
        # get mixture of generated and ground truth data
        #shape_dat = (gen_data - real_data).shape
        alpha = tf.random.normal([self.hparams["batch_size"], 1, 1, 1], 0., 1.)
        mix_data = real_data + alpha * (gen_data - real_data)
        disc_in_gen = [cond_data] + [const_data] + [mix_data]

        with tf.GradientTape() as gp_tape:
            gp_tape.watch(mix_data)
            discriminator_mix = self.discriminator.model(disc_in_gen, training=True)

        # calculate the gradient on the mixture data...
        grads_mix = gp_tape.gradient(discriminator_mix, [mix_data])[0]
        # ... and norm it
        norm = tf.sqrt(tf.reduce_mean(tf.square(grads_mix), axis=[1, 2, 3]))
        gp = tf.reduce_mean((norm - 1.) ** 2)

        return gp
    
    @staticmethod
    def discriminator_loss(real_img, fake_img):
        real_loss = tf.reduce_mean(real_img)
        fake_loss = tf.reduce_mean(fake_img)
        return fake_loss - real_loss


    # Define the loss functions for the generator.
    @staticmethod
    def generator_loss(fake_img):
        return -tf.reduce_mean(fake_img)
        


class HarrisWGAN(AbstractModelClass):
    
    def __init__(self, generator: AbstractModelClass, discriminator: AbstractModelClass, shape_in: List, hparams: dict,
                 varnames_tar: List, savedir: str, expname: str):
        """
        Initialize the HarrisWGANModel class.

        :param generator: The generator model.
        :param discriminator: The discriminator model.
        :param shape_in: The input shape of the model. Note: The last two dimensions must denote the number of coarse-grained predictors 
                         and the number of static high-resolution predictors, respectively.
        :param hparams: Dictionary of custom hyperparameters.
        :param varnames_tar: List of target variable names.
        :param savedir: Drectory to save the model.
        :param expname: The name of the experiment.
        """        
        super().__init__(shape_in, hparams, varnames_tar, savedir, expname)

        self.modelname = "harriswgan"
        
        # set hyperparmaters
        self.set_hparams(hparams)
        # set submodels
        print("Set generator and critic-model...")
        self.generator, self.discriminator = self.set_model(generator, discriminator)
        # set compile and fit options as well as custom objects
        self.set_compile_options()
        self.set_custom_objects(loss=self.compile_options['loss'])
        self.set_fit_options()
        
    def set_compile_options(self):
        """
        Set compile options for the HarrisWGAN model.
        """
        # set optimizers
        if self.hparams["optimizer"].lower() == "adam":
            optimizer = keras.optimizers.Adam
            kwargs_opt = {"beta_1": 0.0, "beta_2": 0.9}
        elif self.hparams["optimizer"].lower() == "rmsprop":
            optimizer = keras.optimizers.RMSprop
            kwargs_opt = {}
        else:
            raise ValueError("'{0}' is not a valid optimizer. Either choose Adam or RMSprop-optimizer")

        self.optimizer = (optimizer(self.discriminator.hparams["lr"], **kwargs_opt), optimizer(self.generator.hparams["lr"], **kwargs_opt))
        
    def get_fit_options(self):
        """
        Get options that will be parsed to the fit-method of the Keras model.
        """
        harriswgan_callbacks = []
        
        if self.hparams["lr_decay"]:
            harriswgan_callbacks.append(LearningRateSchedulerHarrisWGAN(self.get_lr_decay(), verbose=1))
        
        if self.hparams["lcheckpointing"]:
            harriswgan_callbacks.append(ModelCheckpointHarrisWGAN(self._savedir, self._expname, 
                                                                  monitor="val_recon_loss", verbose=1, save_best_only=False, mode="min"))
            
        if self.hparams["learlystopping"]:
            harriswgan_callbacks.append(EarlyStopping(monitor="val_recon_loss", patience=8))
            
        if harriswgan_callbacks is not None:
            return {"callbacks": harriswgan_callbacks}
        else:
            return {}  
        
    def set_model(self, generator, discriminator):
        """
        Instantiate the generator and discriminator models and create the HarrisWGAN model instance.
        :param generator: The generator model.
        :param discriminator: The discriminator model.
        """
        # get relevant shapes for input and output of generator and discriminator
        lo_res_in_shp = list(self._input_shape[:-1] )
        hi_res_in_shp = list(np.array(lo_res_in_shp[:2])*int(np.prod(np.array([4,])))) + [self._input_shape[-1]]
        in_noise_shp = lo_res_in_shp[:2] + [self.hparams["noise_channels"]]    
        out_shp = hi_res_in_shp[:2] + [len(self._varnames_tar)]            
        
        shp_gen = {"lo_res_inputs": lo_res_in_shp, "hi_res_inputs": hi_res_in_shp, "noise_input": in_noise_shp}
        # get generator model
        gen_model = generator(shp_gen, self.hparams["hparams_generator"], self._varnames_tar, hparams_wgan=self.hparams)      
        
        # get discriminator model
        shp_disc = {"lo_res_inputs": lo_res_in_shp, "hi_res_inputs": hi_res_in_shp, "output": out_shp}
        discriminator_model = discriminator(shp_disc, self.hparams["hparams_critic"], self._varnames_tar)
        
        # get hyperparamters of HarrisWGAN only
        hparams_wgan_only = self.hparams.copy()
        hparams_wgan_only.pop("hparams_critic")
        hparams_wgan_only.pop("hparams_generator")
                
        # ...and create HarrisWGAN model instance
        self.model = HarrisWGAN_Model(gen_model, discriminator_model, hparams_wgan_only)

        return gen_model, discriminator_model
    
        
    def get_lr_decay(self):
        """
        Get callable of learning rate scheduler which can be used as callabck in Keras models.
        Exponential decay is applied to change the learning rate from the start to the end value.
        Note that the exponential decay is calculated based on the learning rate of the generator, but applies to both.
        :return: learning rate scheduler
        
        NOTE SL: taken from wgan_model.py
        """
        decay_st, decay_end = self.hparams["decay_start"], self.hparams["decay_end"]
        lr_start, lr_end = self.hparams["hparams_generator"]["lr"], self.hparams["hparams_generator"]["lr_end"]

        if not decay_end > decay_st:
            raise ValueError("Epoch for end of learning rate decay must be large than start epoch. " +
                             "Your values: {0:d}, {1:d})".format(decay_st, decay_end))

        ne_decay = decay_end - decay_st
        # calculate decay rate from start and end learning rate
        decay_rate = 1./ne_decay*np.log(lr_end/lr_start)

        def lr_scheduler(epoch, lr):
            if epoch < decay_st:
                return lr
            elif decay_st <= epoch < decay_end:
                return lr * tf.math.exp(decay_rate)
            elif epoch >= decay_end:
                return lr

        return lr_scheduler

    def plot_model(self, save_dir, **kwargs):
        """
        Plot generator and discriminator model separately.
        :param save_dir: directory under which plots will be saved
        :param kwargs: All keyword arguments valid for tf.keras.utils.plot_model
        
        NOTE SL: taken from wgan_model.py
        """
        k_plot_model(self.generator, os.path.join(save_dir, f"plot_{self._expname}_generator.png"), **kwargs)
        k_plot_model(self.discriminator, os.path.join(save_dir, f"plot_{self._expname}_discriminator.png"), **kwargs)

    def save(self, filepath: str, overwrite: bool = True, include_optimizer: bool = True, save_format: str = None,
             signatures=None, options=None, save_traces: bool = True):
        """
        Save generator and discriminator seperately.
        The parameters of this method are equivalent to Keras.model.save ensuring full functionality.
        :param filepath: path to SavedModel or H5 file to save both models
        :param overwrite: Whether to silently overwrite any existing file at the target location, or provide the user
                          with a manual prompt.
        :param include_optimizer: If True, save optimizer's state together.
        :param save_format: Either `'tf'` or `'h5'`, indicating whether to save the model to Tensorflow SavedModel or
                            HDF5. Defaults to 'tf' in TF 2.X, and 'h5' in TF 1.X.
        :param signatures: Signatures to save with the SavedModel. Applicable to the 'tf' format only.
                           Please see the `signatures` argument in `tf.saved_model.save` for details.
        :param options: (only applies to SavedModel format) `tf.saved_model.SaveOptions` object that specifies options
                        for saving to SavedModel.
        :param save_traces: (only applies to SavedModel format) When enabled, the SavedModel will store the function
                            traces for each layer. This can be disabled, so that only the configs of each layer are
                            stored.  Defaults to `True`. Disabling this will decrease
                            serialization time and reduce file size, but it requires that
                            all custom layers/models implement a `get_config()` method.
        :return: -
        """
        generator_path, discriminator_path = os.path.join(filepath, "{0}_generator_last".format(self._expname)), \
                                      os.path.join(filepath, "{0}_discriminator_last".format(self._expname))
        self.generator.save(generator_path, overwrite, include_optimizer, save_format, signatures, options, save_traces)
        self.discriminator.save(discriminator_path, overwrite, include_optimizer, save_format, signatures, options, save_traces)
                          
    def set_hparams_default(self):
        """
        Note: Hyperparameter defaults taken from 1) https://github.com/ECMWFCode4Earth/tesserugged/blob/master/dev/gan/dsrnngan/local_config.yaml and 2) https://github.com/ECMWFCode4Earth/tesserugged/blob/master/dev/gan/dsrnngan/models.py
        """
        self.hparams_default = {"batch_size": 2, "nepochs": 30, "lr_decay": False, "decay_start": 3, "decay_end": 20, "stream_mode": "lo_input",
                                "l_embed": False, "ds_steps": [4,], "d_steps": 5, "recon_weight": 1000., "gp_weight": 10., "optimizer": "adam", 
                                "lcheckpointing": True, "learlystopping": False, "recon_loss": "ensmeanMSE", "ensemble_size": 8,  
                                "noise_channels": 4, "hparams_generator": {}, "hparams_critic": {} }
        
    def load_inference_model(self, model_dir, format="tf"):
            
        # construct directories to generator- and critic model from model directory
        model_dir = Path(model_dir)

        expname = model_dir.name
        suffix = expname.split("_")[-1]

        fname_suffix = ".h5" if format == "h5" else ""

        gen_dir = model_dir.joinpath(expname.replace(suffix, f"generator_{suffix}{fname_suffix}"))

        # load saved models
        generator = keras.models.load_model(gen_dir, compile=False)
        
        # construct noise genartor required for ensemble 
        noise_gen = NoiseGenerator(list(generator.get_layer(name='noise_input').input_shape[0][1:]), self.hparams["batch_size"])

        hparams_wgan_only = self.hparams.copy()
        hparams_wgan_only.pop("hparams_critic")
        hparams_wgan_only.pop("hparams_generator")
        
        # get construct model for inference exposing predict-method
        # Note the predict-step makes use of the generator only. Thus, the critic model is not needed here
        wgan_model = HarrisWGAN_Model(generator, None, hparams_wgan_only, noise_gen=noise_gen)
        
        return wgan_model

            
class LearningRateSchedulerHarrisWGAN(LearningRateSchedulerWGAN):
    """Note SL: taken from wgan_model.py"""
    def __init__(self, schedule, verbose=0):
        super(LearningRateSchedulerWGAN, self).__init__(schedule, verbose)

    def on_epoch_begin(self, epoch, logs=None):
        if not hasattr(self.model, "g_optimizer"):
            raise AttributeError('Model must have a "g_optimizer" for optimizing the generator.')

        if not hasattr(self.model, "c_optimizer"):
            raise AttributeError('Model must have a "c_optimizer" for optimizing the discriminator.')

        if not (hasattr(self.model.g_optimizer, "lr") and hasattr(self.model.c_optimizer, "lr")):
            raise ValueError('Optimizer for generator and discriminator must both have a "lr" attribute.')
        try:  # new API
            lr_g, lr_c = float(K.get_value(self.model.g_optimizer.lr)), \
                         float(K.get_value(self.model.c_optimizer.lr))
            lr_g, lr_c = self.schedule(epoch, lr_g), self.schedule(epoch, lr_c)
        except TypeError:  # Support for old API for backward compatibility
            raise NotImplementedError("WGAN learning rate schedule is not compatible with old API. Update TF Keras.")

        if not (isinstance(lr_g, (tf.Tensor, float, np.float32, np.float64)) and
                isinstance(lr_c, (tf.Tensor, float, np.float32, np.float64))):
            raise ValueError('The output of the "schedule" function '
                             f'should be float. Got: {lr_g} (generator) and {lr_c} (discriminator)' )
        if isinstance(lr_g, tf.Tensor) and not lr_g.dtype.is_floating \
           and isinstance(lr_c, tf.Tensor) and lr_c.dtype.is_floating:
            raise ValueError(
                f'The dtype of `lr_g` and `lr_c` Tensor should be float. Got: {lr_g.dtype} (generator)'
                f'and {lr_c.dtype} (discriminator)' )
        # set updated learning rate
        K.set_value(self.model.g_optimizer.lr, K.get_value(lr_g))
        K.set_value(self.model.c_optimizer.lr, K.get_value(lr_c))
        if self.verbose > 0:
            print(f'\nEpoch {epoch + 1}: LearningRateScheduler setting learning '
                  f'rate for generator to {lr_g}, for discriminator to {lr_c}.')

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        logs['lr_generator'] = K.get_value(self.model.g_optimizer.lr)
        logs['lr_discriminator'] = K.get_value(self.model.c_optimizer.lr)


class ModelCheckpointHarrisWGAN(ModelCheckpoint):
    """Note SL: taken from wgan_model.py"""
    def __init__(self, filepath, expname, monitor='val_loss', verbose=0, save_best_only=False, save_weights_only=False,
                 mode='auto', save_freq="epoch", options=None, **kwargs):
        super(ModelCheckpointHarrisWGAN, self).__init__(filepath,  monitor, verbose, save_best_only,
                                                  save_weights_only, mode, save_freq, options=options, **kwargs)
        self._expname = expname

    def _save_model(self, epoch, batch, logs):
        """Saves the model.
        ML: The source-code is largely identical to Keras v2.6.0 implementation except that two models,
            the discriminator and the generator, are saved separately in filepath_gen and filepath_discriminator (see below).
            Modified source-code is envelopped between 'ML S' and 'ML E'-comment strings.

        Args:
            epoch: the epoch this iteration is in.
            batch: the batch this iteration is in. `None` if the `save_freq`
              is set to `epoch`.
            logs: the `logs` dict passed in to `on_batch_end` or `on_epoch_end`.
        """
        logs = logs or {}

        if isinstance(self.save_freq, int) or self.epochs_since_last_save >= self.period:
            # Block only when saving interval is reached.
            logs = tf_utils.sync_to_numpy_or_python_type(logs)
            self.epochs_since_last_save = 0
            filepath = self._get_file_path(epoch, batch, logs)
            # ML S
            if self.save_best_only:
                add_str = "best"
            else:
                add_str = f"epoch{epoch:05d}"
            filepath_gen = os.path.join(filepath, f"{self._expname}_generator_{add_str}")
            filepath_discriminator = os.path.join(filepath, f"{self._expname}_discriminator_{add_str}")
            # ML E

            try:
                if self.save_best_only:
                    current = logs.get(self.monitor)
                    if current is None:
                        logging.warning('Can save best model only with %s available, skipping.', self.monitor)
                    else:
                        if self.monitor_op(current, self.best):
                            if self.verbose > 0:
                                print('\nEpoch %05d: %s improved from %0.5f to %0.5f,'
                                      ' saving model to %s' % (epoch + 1, self.monitor,
                                                               self.best, current, filepath))
                            self.best = current
                            # ML S
                            if self.save_weights_only:
                                self.model.generator.save_weights(
                                    filepath_gen, overwrite=True, options=self._options)
                                self.model.discriminator.save_weights(
                                    filepath_discriminator, overwrite=True, options=self._options)
                            else:
                                self.model.generator.save(filepath_gen, overwrite=True, options=self._options)
                                self.model.discriminator.save(filepath_discriminator, overwrite=True, options=self._options)
                            # ML E
                        else:
                            if self.verbose > 0:
                                print('\nEpoch %05d: %s did not improve from %0.5f' %
                                      (epoch + 1, self.monitor, self.best))
                else:
                    if self.verbose > 0:
                        print('\nEpoch %05d: saving model to %s' % (epoch + 1, filepath))
                    # ML S
                    if self.save_weights_only:
                        self.model.generator.save_weights(
                            filepath_gen, overwrite=True, options=self._options)
                        self.model.discriminator.save_weights(
                            filepath_discriminator, overwrite=True, options=self._options)
                    else:
                        self.model.generator.save(filepath_gen, overwrite=True, options=self._options)
                        self.model.discriminator.save(filepath_discriminator, overwrite=True, options=self._options)
                    # ML E
                self._maybe_remove_file()
            except IsADirectoryError as e:  # h5py 3.x
                raise IOError('Please specify a non-directory filepath for'  
                              'ModelCheckpoint. Filepath used is an existing directory: {}'.format(filepath))
            except IOError as e:  # h5py 2.x
                # `e.errno` appears to be `None` so checking the content of `e.args[0]`.
                if 'is a directory' in str(e.args[0]).lower():
                    raise IOError('Please specify a non-directory filepath for '
                                  'ModelCheckpoint. Filepath used is an existing directory: {}'.format(filepath))
                # Re-throw the error for any other causes.
                raise e

####################################################################################
####################################################################################
# some classes and methods that are used in the original Harris GAN implementation
####################################################################################
####################################################################################
class ReflectionPadding2D(Layer):
    def __init__(self, padding=(1, 1), **kwargs):
        self.padding = tuple(padding)
        super(ReflectionPadding2D, self).__init__(**kwargs)

    def compute_output_shape(self, s):
        return (
            s[0],
            None if s[1] is None else s[1]+2*self.padding[0],
            None if s[2] is None else s[2]+2*self.padding[1],
            s[3]
        )

    def call(self, x):
        i_pad, j_pad = self.padding
        return tf.pad(x, [[0, 0], [i_pad, i_pad], [j_pad, j_pad], [0, 0]], 'REFLECT')


class SymmetricPadding2D(Layer):
    def __init__(self, padding=(1, 1), **kwargs):
        self.padding = tuple(padding)
        super(SymmetricPadding2D, self).__init__(**kwargs)

    def compute_output_shape(self, s):
        return (
            s[0],
            None if s[1] is None else s[1]+2*self.padding[0],
            None if s[2] is None else s[2]+2*self.padding[1],
            s[3]
        )

    def call(self, x):
        i_pad, j_pad = self.padding
        return tf.pad(x, [[0, 0], [i_pad, i_pad], [j_pad, j_pad], [0, 0]], 'SYMMETRIC')

class Conv2DPadding(Layer):
    def __init__(self, filters, kernel_size, stride, padding, dilations):
        super(Conv2DPadding, self).__init__()
        self.filters = filters
        self.kernel_size = kernel_size
        self.stride = stride
        self.padding = padding
        self.dilation = dilations
        if not isinstance(dilations, int):
            # padding calculation in build() would need to be adjusted to handle a tuple/list
            raise NotImplementedError("Only integer dilation is supported.")
        if padding is None:
            raise ValueError("padding should not be None")

    def build(self, x):
        if self.padding in ('reflect', 'symmetric'):
            pad = tuple((self.dilation*(s-1))//2 for s in self.kernel_size)  # only works if s is odd, or dilation is even
            if self.padding == 'reflect':
                self.padref = ReflectionPadding2D(padding=pad)
            elif self.padding == 'symmetric':
                self.symref = SymmetricPadding2D(padding=pad)
            self.convval = Conv2D(filters=self.filters,
                                  kernel_size=self.kernel_size,
                                  strides=(self.stride, self.stride),
                                  padding='valid',
                                  dilation_rate=self.dilation)
        else:
            self.convsam = Conv2D(filters=self.filters,
                                  kernel_size=self.kernel_size,
                                  strides=(self.stride, self.stride),
                                  padding='same',
                                  dilation_rate=self.dilation)

    def call(self, x):
        if self.padding in ('reflect', 'symmetric'):
            if self.padding == 'reflect':
                x = self.padref(x)
            elif self.padding == 'symmetric':
                x = self.symref(x)
            return self.convval(x)
        else:  # same
            return self.convsam(x)
        
def residual_block(x, filters, conv_size=(3, 3), stride=1, dilations=1, relu_alpha=0.2, padding=None):
    in_channels = int(x.shape[-1])
    x_in = x

    if stride > 1:
        x_in = AveragePooling2D(pool_size=(stride, stride))(x_in)
    if (filters != in_channels):
        x_in = Conv2D(filters=filters, kernel_size=(1, 1))(x_in)

    # first block of activation and 3x3 convolution (possibly strided, although we don't use this)
    x = LeakyReLU(relu_alpha)(x)
    x = Conv2DPadding(filters=filters, kernel_size=conv_size, stride=stride, dilations=dilations, padding=padding)(x)

    # second block of activation and 3x3 unstrided convolution
    x = LeakyReLU(relu_alpha)(x)
    x = Conv2DPadding(filters=filters, kernel_size=conv_size, stride=1, dilations=dilations, padding=padding)(x)
    
    # skip connection
    x = Add()([x, x_in])

    return x


def const_upscale_block(const_input, steps, filters):
    # Map (N x kH x kW x C) to (N x H x W x f), where k is downscaling factor
    const_output = const_input
    for step in steps:
        const_output = Conv2D(filters=filters, kernel_size=(step, step), strides=step, padding="valid", activation="relu")(const_output)
    return const_output


def wasserstein_loss(y_true, y_pred):
    #return 1.
    return K.mean(y_true * y_pred, axis=-1)

def ensmean_MSE(y_true, y_pred):
    pred_mean = tf.squeeze(tf.reduce_mean(y_pred, axis=0), axis=-1)
    y_true_squ = tf.squeeze(y_true, axis=-1)
    return tf.reduce_mean(tf.math.squared_difference(pred_mean, y_true_squ))

def CL_chooser(CLtype):
    if CLtype != "ensmeanMSE":
        raise NotImplementedError(f"{CLtype = } not implemented, only CLtype = 'ensmeanMSE' is available!")
        
    return {
        # "CRPS": sample_crps,
        # "CRPS_phys": sample_crps_phys,
        "ensmeanMSE": ensmean_MSE,
        # "ensmeanMSE_phys": ensmean_MSE_phys#
    }[CLtype]

In [63]:
from unet_model import Sha_UNet, DeepRU_UNet
from wgan_model import WGAN, Critic_Simple
from other_utils import to_list

class ModelEngine(object):
    """
    Class to get and instantiate known models.
    To add new models, please adapt known_models accordingly.
    General info:
    The key value of the implemented model should be based on keras.Model and should expect 'hparams', 'exp_name' and
    'model_savedir' as arguments for initialization. Further keyword arguments are possible.
    For composite models (e.g. GANs):
    The key value should be a tuple whose first element constitutes the composited model (based on keras.Model).
    The following arguments should then be model construction objects to define the components of the composite model.
    Example: WGAN is a composite model consisting of a generator and critic (see wgan_model.py).
             Thus, the derived class should be the first element of the tuple.
             The model constructions of the generator (e.g. a U-Net) and the critic must then constitute the second and
             third element of the tuple, i.e. {"wgan": (WGAN, Sha_UNet, Critic_Simple).
    """

    known_models = {"sha_unet": (Sha_UNet,),
                    "deepru": (DeepRU_UNet,),
                    "sha_wgan": (WGAN, Sha_UNet, Critic_Simple),
                    "harris_wgan": (HarrisWGAN, GeneratorHarris, DiscriminatorHarris)}
    
    long_names = ["Sha U-Net", "DeepRU", "Sha WGAN", "Harris WGAN"]
    
    assert len(known_models) == len(long_names), f"Conflicting number of known_models ({len(known_models)})" + \
                                                 f" and long_names ({len(long_names)})."

    def __init__(self, model_name: str):
        """
        Initialize the model if known.
        :param model_name: name of the model (must match any key of self.known_models)
        Hint: Pass help to get an overview of the available models.
        """
        self.modelname = model_name
        self.model = self.known_models[self.modelname]
        self.model_longname = self.long_names[list(self.known_models.keys()).index(self.modelname)]

    def __call__(self, shape_in, varnames_tar, hparams_dict, save_dir, expname, **kwargs):
        """
        Instantiate the model with some required arguments.
        """
        model_list = to_list(self.model)
        target_model = model_list[0]
        model_args = {"shape_in": shape_in, "varnames_tar": varnames_tar, "hparams": hparams_dict,
                      "savedir": save_dir, "expname": expname, **kwargs}

        try:
            if len(model_list) == 1:
                model = target_model(**model_args)
            else:
                submodels = model_list[1:]
                model = target_model(*submodels, **model_args)

                # Fix to ensure that correct modelname is set
                model.modelname = self.modelname
        except Exception as e:
            err_str = str(e)
            raise RuntimeError(f"Failed to instantiate the model. The following error occured: \n {err_str}")

        return model

    @property
    def modelname(self):
        return self._modelname

    @modelname.setter
    def modelname(self, model_name):

        help_str = self._get_help_str()
        model_name_local = model_name.lower()

        if model_name_local in self.known_models.keys():
            self._modelname = model_name_local
        elif model_name_local == "help":
            print(help_str)
            self._modelname = None
        else:
            raise ValueError(f"Model '{model_name}' is unknown. Please specify a known model. {help_str}")

    def _get_help_str(self):
        """
        Create help-string listing all known models.
        """
        return f"Known models are: {', '.join(list(self.known_models.keys()))}"

In [3]:
model_dir = Path("/p/home/jusers/langguth1/juwels/downscaling_maelstrom/downscaling_benchmark/trained_models/t2m/complete/harris_wgan_tests/" +
                 "harris_wgan_recon2000/harris_wgan_recon2000_generator_epoch00008")
data_dir = Path("/p/scratch/deepacf/maelstrom/maelstrom_data/ap5/downscaling_benchmark_dataset/benchmark_t2m/dataset/coarse_input/")
model_basedir = model_dir.parents[0]

dataset="benchmark_t2m"

js_norm = model_basedir.joinpath("norm.json")
model_config_js = model_basedir.joinpath("config_harris_wgan.json")
dataset_config_js = model_basedir.joinpath("config_ds_benchmark_t2m.json")

In [57]:
with open(dataset_config_js) as dsf:
    print(f"Read dataset configuration file '{dataset_config_js}'.")
    ds_dict = js.load(dsf)
    
with open(model_config_js) as dsm:
    print(f"Read dataset configuration file '{model_config_js}'.")
    hparams_dict = js.load(dsm)
    
data_norm = ZScore(ds_dict["norm_dims"])
data_norm.read_norm_from_file(js_norm)

Read dataset configuration file '/p/home/jusers/langguth1/juwels/downscaling_maelstrom/downscaling_benchmark/trained_models/t2m/complete/harris_wgan_tests/harris_wgan_recon2000/config_ds_benchmark_t2m.json'.
Read dataset configuration file '/p/home/jusers/langguth1/juwels/downscaling_maelstrom/downscaling_benchmark/trained_models/t2m/complete/harris_wgan_tests/harris_wgan_recon2000/config_harris_wgan.json'.


In [58]:
hparams_dict["batch_size"] = 36
ds_dict["batch_size"] = 36

In [59]:
#tfds_train, train_info = prepare_dataset(data_dir, "benchmark_t2m", ds_dict, hparams_dict, "train", norm_obj=data_norm, shuffle=True, lrepeat=True, drop_remainder=True)
#tfds_val, val_info = prepare_dataset(data_dir, "benchmark_t2m", ds_dict, hparams_dict, "val", norm_obj=data_norm, shuffle=False, lrepeat=True, drop_remainder=False)
tfds_test, test_info = prepare_dataset(data_dir, "benchmark_t2m", ds_dict, hparams_dict, "test", norm_obj=data_norm, shuffle=False, lrepeat=False, drop_remainder=True)

Selected stream mode for test dataset: lo_input


In [ ]:
print(hparams_dict)
print(ds_dict)

In [66]:
model_dir = Path("/p/home/jusers/langguth1/juwels/downscaling_maelstrom/downscaling_benchmark/trained_models/t2m/complete/harris_wgan_tests/harris_wgan_recon2000/harris_wgan_recon2000_epoch00003/")

model_instance = ModelEngine("harris_wgan")

print(hparams_dict)
model = model_instance([1, 1, 1, 1], "dummy", hparams_dict, "~/test/", "dummy")

trained_model = model.load_inference_model(model_dir)

{'batch_size': 36, 'nepochs': 10, 'lr_decay': True, 'decay_start': 2, 'decay_end': 5, 'stream_mode': 'lo_input', 'l_embed': False, 'ds_steps': [4], 'd_steps': 5, 'recon_weight': 2000.0, 'gp_weight': 10.0, 'optimizer': 'adam', 'lcheckpointing': True, 'learlystopping': False, 'recon_loss': 'ensmeanMSE', 'ensemble_size': 8, 'noise_channels': 4, 'hparams_generator': {'lr': 1e-05, 'lr_end': 1e-06, 'channels_start': 128}, 'hparams_critic': {'lr': 5e-07}}
Set generator and critic-model...


In [67]:
y_pred = trained_model.predict(tfds_test, verbose=1, steps = 5)

2024-09-19 13:55:14.670311: I tensorflow/stream_executor/cuda/cuda_dnn.cc:369] Loaded cuDNN version 8301


5/5 [==============================] - 8s 640ms/step


2024-09-19 13:55:21.912642: W tensorflow/core/kernels/data/cache_dataset_ops.cc:768] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset  will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`. You should use `dataset.take(k).cache().repeat()` instead.


In [ ]:
print(np.shape(y_pred))

In [ ]:
y_pred = np.squeeze(np.mean(y_pred, axis=-1))

In [ ]:
import xarray as xr

tar_varname = test_info["all_predictands"][0]
print(f"Variable {tar_varname} serves as ground truth data.")

# get ground truth data
ds_test = xr.open_dataset(test_info["file"])
# rename coordinates and dimensions of target data for consistency
dims_new = [dim.replace("_tar", "") for dim in ds_test[tar_varname].dims]
ds_test = ds_test.rename({old: new for old, new in zip(ds_test[tar_varname].dims, dims_new) if old != new})
coords, dims = ds_test[tar_varname].squeeze().coords, ds_test[tar_varname].squeeze().dims
print(ds_test)

In [ ]:
from other_utils import convert_to_xarray, finditem

y_pred = convert_to_xarray(y_pred, data_norm, tar_varname, coords, dims, finditem(hparams_dict, "z_branch", False))

In [ ]:
y_ref = ds_test["t_2m_tar"]

In [ ]:
from scores_class import Scores

score_engine = Scores(y_pred, y_ref, ["time", "rlat", "rlon"])

In [ ]:
print(score_engine("grad_amplitude").mean())

# Enable resume training

The below implemented solution has been obtained from [here](https://stackoverflow.com/questions/49503748/save-and-load-model-optimizer-state).
Note that the `weights`-attribute is not available with more recent Keras version, whereas the `variables()`-method can then be used to get the optimizer state.


In [ ]:
import os, sys
#sys.path.append("../models/")
#sys.path.append("../utils/")
#sys.path.append("../handle_data/")

from pathlib import Path
from typing import List, Tuple, Union, Dict
from collections import OrderedDict
import types
import json as js
import numpy as np
import glob 
import pickle
from abstract_model_class import AbstractModelClass
import tensorflow as tf
import tensorflow.keras as keras
from tensorflow.python.keras.utils import tf_utils
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from tensorflow.keras.layers import Input, concatenate, LeakyReLU, UpSampling2D, Layer, BatchNormalization, Conv2D, Add, AveragePooling2D, GlobalAveragePooling2D, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.utils import plot_model as k_plot_model
from tensorflow.keras import backend as K
from custom_losses import get_custom_loss
from wgan_model import LearningRateSchedulerWGAN
from tensorflow.python.platform import tf_logging as logging
from handle_data_class import prepare_dataset
from all_normalizations import ZScore
from abstract_data_normalization import Normalize

In [ ]:
print(tf.__version__)

In [68]:
list_or_tuple = Union[List, Tuple]
                    
def save_opt_weights(optimizer, filepath):
    symbolic_weights = getattr(optimizer, 'weights')
    if symbolic_weights:
        weight_values = K.batch_get_value(symbolic_weights)
        
        with open(filepath, 'wb') as f:
            pickle.dump(weight_values, f)
    else:
        raise ValueError(f"Failed to deduce weight from optimizer")

class GeneratorHarris(AbstractModelClass):
    # content based on original generator function from models.py (Harris repo)
    # structure based on Critic_Simple from wgan_model.py
    def __init__(self, shape_in: List, hparams: dict, varnames_tar: List):
        # Pass empty savedir- and expname-arguments since this is not a stand-alone model
        super().__init__(shape_in, hparams, varnames_tar, "", "")
        
        # set submodels
        self.set_hparams(hparams)
        self.set_model()
        
    def set_model(self):
        ds_steps = self.hparams["ds_steps"]
        filters_gen = self.hparams["channels_start"]
        kernel = self.hparams["kernel"]
        relu_alpha = self.hparams["relu_alpha"]
        padding = self.hparams["padding"]
        # Network inputs
        # low resolution condition
        generator_input = Input(shape=self._input_shape["lo_res_inputs"], name="lo_res_inputs")
        # constant fields
        const_input = Input(shape=self._input_shape["hi_res_inputs"], name="hi_res_inputs")

        # Convolve constant fields down to match other input dimensions
        upscaled_const_input = const_upscale_block(
            const_input, steps=ds_steps, filters=filters_gen
        )
        # noise
        noise_input = Input(shape=self._input_shape["noise_input"], name="noise_input")
        # Concatenate all inputs together
        generator_output = concatenate(
            [generator_input, upscaled_const_input, noise_input]
        )

        # Pass through 3 residual blocks
        for ii in range(3):
            generator_output = residual_block(
                generator_output,
                filters=filters_gen,
                conv_size=kernel,
                stride=1,
                relu_alpha=relu_alpha,
                padding=padding,
            )

        # Upsampling from low-res to high-res with alternating residual blocks
        # In the paper, this was [2*filters_gen, filters_gen] for steps of 5 and 2
        block_channels = [2 * filters_gen] * (len(ds_steps) - 1) + [filters_gen]
        for ii, step in enumerate(ds_steps):
            generator_output = UpSampling2D(size=(step, step), interpolation="bilinear")(
                generator_output
            )

            generator_output = residual_block(
                generator_output,
                filters=block_channels[ii],
                conv_size=kernel,
                stride=1,
                relu_alpha=relu_alpha,
                padding=padding,
            )

        # Concatenate with original size constants field
        generator_output = concatenate([generator_output, const_input])

        # Pass through 3 residual blocks
        for ii in range(3):
            generator_output = residual_block(
                generator_output,
                filters=filters_gen,
                conv_size=kernel,
                stride=1,
                relu_alpha=relu_alpha,
                padding=padding,
            )

        # Output layer
        actv_out = "linear"
        generator_output = Conv2D(
            filters=1, kernel_size=(1, 1), activation=actv_out, name="output"
        )(generator_output)
        
        self.model = Model(
            inputs=[generator_input, const_input, noise_input],
            outputs=generator_output,
            name="gen",
        )
                   
    def set_compile_options(self):
        raise RuntimeError(f"Generator model is supposed to be part of a composite model such as WGAN, but not as standalone model for training.")
        
    def set_fit_options(self):
        raise RuntimeError(f"Generator model is supposed to be part of a composite model such as WGAN, but not as standalone model for training.")
        
    def set_hparams_default(self):
        """
        Note: hyperparameter defaults of generator and critic model must be set in the respective model classes whose instances are just parsed here.
        """
        self.hparams_default = {"channels_start": 128, "activation": "leaky_relu", "kernel": (3, 3), "stride": (2, 2), "lr": 1.e-5, 
                                "ds_steps": [4,], "padding": "reflect", "relu_alpha": 0.2, "lr_end": 1.e-05}


class CriticHarris(AbstractModelClass):
    # content based on original discriminator function from models.py (Harris repo)
    # structure based on Critic_Simple from wgan_model.py
    def __init__(self, shape_in: List, hparams: dict, varnames_tar: List):
        # Pass empty savedir- and expname-arguments since this is not a stand-alone model
        super().__init__(shape_in, hparams, varnames_tar, "", "")
        
        # set submodels
        self.set_hparams(hparams)
        self.set_model()
        
    def set_model(self):
        ds_steps = self.hparams["ds_steps"]
        filters_critic = self.hparams["channels_start"]
        kernel = self.hparams["kernel"]
        relu_alpha = self.hparams["relu_alpha"]
        padding = self.hparams["padding"]
        # Network inputs
        # low resolution condition
        generator_input = Input(shape=self._input_shape["lo_res_inputs"], name="lo_res_inputs")
        # constant fields
        const_input = Input(shape=self._input_shape["hi_res_inputs"], name="hi_res_inputs")
        # target image
        generator_output = Input(shape=self._input_shape["output"], name="output")

        # convolve down constant fields to match ERA
        lo_res_const_input = const_upscale_block(
            const_input, steps=ds_steps, filters=filters_critic
        )

        # concatenate constants to lo-res input
        lo_res_input = concatenate([generator_input, lo_res_const_input])

        # concatenate constants to hi-res input
        hi_res_input = concatenate([generator_output, const_input])

        # encode inputs using residual blocks
        # In the paper, this was [filters_disc, 2*filters_disc] for steps of 5 and 2
        block_channels = [filters_critic] * (len(ds_steps) - 1) + [2 * filters_critic]

        for ii, step in enumerate(ds_steps):
            lo_res_input = residual_block(
                lo_res_input,
                filters=block_channels[ii],
                conv_size=kernel,
                stride=1,
                relu_alpha=relu_alpha,
                padding=padding,
            )
            hi_res_input = Conv2D(
                filters=block_channels[ii],
                kernel_size=(step, step),
                strides=step,
                padding="valid",
                activation="relu",
            )(hi_res_input)

            hi_res_input = residual_block(
                hi_res_input,
                filters=block_channels[ii],
                conv_size=kernel,
                stride=1,
                relu_alpha=relu_alpha,
                padding=padding,
            )

        # concatenate hi- and lo-res inputs channel-wise before passing through critic
        critic_input = concatenate([lo_res_input, hi_res_input])

        # encode in residual blocks
        critic_input = residual_block(
            critic_input,
            filters=filters_critic,
            conv_size=kernel,
            stride=1,
            relu_alpha=relu_alpha,
            padding=padding,
        )

        # critic output
        critic_output = GlobalAveragePooling2D()(critic_input)
        critic_output = Dense(64, activation="relu")(critic_output)
        critic_output = Dense(1, name="critic_output")(critic_output)

        self.model = Model(
            inputs=[generator_input, const_input, generator_output],
            outputs=critic_output,
            name="critic",
        )
                   
    def set_compile_options(self):
        raise RuntimeError(f"critic model is supposed to be part of a composite model such as WGAN, but not as standalone model for training.")
        
    def set_fit_options(self):
        raise RuntimeError(f"critic model is supposed to be part of a composite model such as WGAN, but not as standalone model for training.")
        
    def set_hparams_default(self):
        """
        Note: hyperparameter defaults of generator and critic model must be set in the respective model classes whose instances are just parsed here.
        """
        self.hparams_default = {"channels_start": 512, "activation": "leaky_relu", "kernel": (3, 3), "stride": (2, 2), 
                                "lr": 1.e-5, "ds_steps": [4,], "padding": "reflect", "relu_alpha": 0.2, "lr_end": 1.e-06}

    
class NoiseGenerator(object):
    """Used for the Generator to generate the random noise input"""
    def __init__(self, noise_shapes, batch_size=32, random_seed=None):
        self.noise_shapes = noise_shapes
        self.batch_size = batch_size
        self.prng = np.random.RandomState(seed=random_seed)

    def noise(self, shape, mean, std):
        shape = [self.batch_size] + shape
        n = self.prng.randn(*shape).astype(np.float32)
        # n = np.zeros(shape, dtype=np.float32)
        if std != 1.0:
            n *= std
        if mean != 0.0:
            n += mean
        return n
    
    def __call__(self, mean=0.0, std=1.0):
        return self.noise(self.noise_shapes, mean, std)
    
    

class HarrisWGAN_Model(keras.Model):
    def __init__(self, generator, critic, hparams, expname):
        super().__init__()
        self.generator = generator
        self.critic = critic
        self.hparams = hparams
        self._expname = expname
        
    def compile(self, optimizer, loss, **kwargs):
        super().compile(**kwargs)
        self.c_optimizer, self.g_optimizer = optimizer
        
        # losses        
        self.noise_gen = NoiseGenerator(
            self.generator._input_shape["lo_res_inputs"][:2]+[self.hparams["noise_channels"]],
            self.hparams["batch_size"]*(self.hparams["d_steps"] + 1)
        )

        # losses
        self.critic_loss = self.critic_loss #get_custom_loss("critic")
        self.critic_gen_loss = self.generator_loss #get_custom_loss("critic_generator")
        self.recon_loss = CL_chooser(self.hparams["recon_loss"])
        
    @tf.function    
    def train_step(self, data_iter: Dict, embed=None) -> OrderedDict:
        inputs, outputs = data_iter
        cond = inputs["lo_res_inputs"]
        const = inputs["hi_res_inputs"]
        if self.hparams["ensemble_size"] is None:
            noise = self.noise_gen()
        else:
            # ensemble stacked in an additional dimension at the end
            noise = tf.stack([self.noise_gen() for _ in range(self.hparams["ensemble_size"] + 1)], axis=-1)
        sample = outputs["output"]

        # train critic
        for i in range(self.hparams["d_steps"]):
            with tf.GradientTape() as tape_critic:
                
                ist, ie = i * self.hparams["batch_size"], (i + 1) * self.hparams["batch_size"]
                cond_iter = cond[ist:ie, ...]
                const_iter = const[ist:ie, ...]
                sample_iter = sample[ist:ie, ...]
                noise_iter = noise[ist:ie, ..., 0] # only take the first ensemble member

                gen_in = [cond_iter] + [const_iter] + [noise_iter]
                gen_out = self.generator.model(gen_in, training=True)
                critic_in_gen = [cond_iter] + [const_iter] + [gen_out]
                critic_in_gt = [cond_iter] + [const_iter] + [sample_iter]
                
                # calculate critic for both, the real and the generated data
                critic_gen = self.critic.model(critic_in_gen, training=True)
                critic_gt = self.critic.model(critic_in_gt, training=True)
                # calculate the loss (incl. gradient penalty)
                c_loss = self.critic_loss(critic_gt, critic_gen)
                #gp = GradientPenalty()([sample_iter, gen_out])
                gp = self.gradient_penalty(sample_iter, gen_out, cond_iter, const_iter)
                d_loss = c_loss + self.hparams["gp_weight"] * gp

            # calculate gradients and update critic
            d_gradient = tape_critic.gradient(d_loss, self.critic.trainable_variables)
            self.c_optimizer.apply_gradients(zip(d_gradient, self.critic.trainable_variables))

        # train generator
        with tf.GradientTape() as tape_generator:
            # generate (downscaled) data
            cond_iter = cond[-self.hparams["batch_size"]:, ...]
            const_iter = const[-self.hparams["batch_size"]:, ...]
            noise_iter = noise[-self.hparams["batch_size"]:, ...]
            sample_iter = sample[-self.hparams["batch_size"]:, ...]

            # train generator for each ensemble member
            noise_iter_k = noise_iter[..., 0]
            gen_in = [cond_iter] + [const_iter] + [noise_iter_k]
            gen_data = self.generator.model(gen_in, training=True)
            gen_data_list = [gen_data]
            if self.hparams["ensemble_size"] is not None:
                gen_iter_list = []
                for k in range(self.hparams["ensemble_size"]):
                    noise_iter_k = noise_iter[..., k+1]
                    gen_in = [cond_iter] + [const_iter] + [noise_iter_k]
                    gen_data_iter = self.generator.model(gen_in, training=True)
                    gen_iter_list.append(gen_data_iter)
                    
                gen_data_list.append(tf.stack(gen_iter_list))
            
            critic_in_gen = [cond_iter] + [const_iter] + [gen_data]
            critic_gen = self.critic.model(critic_in_gen, training=True)

            # critic loss for generator
            cg_loss = self.critic_gen_loss(critic_gen)
            # content loss term
            cl_loss = self.recon_loss(sample_iter, gen_data_list[-1])
            # combined loss for generator
            g_loss = cg_loss + cl_loss*self.hparams["recon_weight"]


        g_gradient = tape_generator.gradient(g_loss, self.generator.trainable_variables)
        self.g_optimizer.apply_gradients(zip(g_gradient, self.generator.trainable_variables))

        return OrderedDict(
            [
                ("c_loss", c_loss),
                ("gp_loss", self.hparams["gp_weight"] * gp),
                ("d_loss", d_loss),
                ("cg_loss", cg_loss),
                #("recon_loss", cl_loss),
                ("recon_loss", cl_loss * self.hparams["recon_weight"]),
                ("g_loss", g_loss)
            ]
        )
            

    def test_step(self, val_iter: tf.data.Dataset) -> OrderedDict:
        """
        Implement step to test trained generator on validation data
        :param val_iter: Tensorflow Dataset with validation data
        :return: dictionary with reconstruction loss on validation data
        
        NOTE SL: taken from wgan_model.py
        """
        inputs, outputs = val_iter
        cond = inputs["lo_res_inputs"]
        const = inputs["hi_res_inputs"]
        sample = outputs["output"]
        if self.hparams["ensemble_size"] is None:
            noise = self.noise_gen()
        else:
            # ensemble stacked in an additional dimension at the end
            noise = tf.stack([self.noise_gen() for _ in range(self.hparams["ensemble_size"] + 1)], axis=-1)
        
        noise_iter = noise[0:self.hparams["batch_size"]:, ...]
        noise_0 = noise_iter[..., 0]
        gen_in = [cond] + [const] + [noise_0]
        gen_data = self.generator.model(gen_in, training=True)
        gen_data_list = [gen_data]
        if self.hparams["ensemble_size"] is not None:
            gen_iter_list = []
            for k in range(self.hparams["ensemble_size"]):
                noise_k = noise_iter[..., k+1]
                gen_in = [cond] + [const] + [noise_k]
                gen_data_k = self.generator.model(gen_in, training=True)
                gen_iter_list.append(gen_data_k)

            gen_data_list.append(tf.stack(gen_iter_list))

        critic_in_gen = [cond] + [const] + [gen_data]
        critic_gen = self.critic.model(critic_in_gen, training=True)

        # critic loss for generator
        cg_loss = self.critic_gen_loss(critic_gen)
        # content loss term
        cl_loss = self.recon_loss(sample, gen_data_list[-1])

        return OrderedDict([
            ("cg_loss", cg_loss),
            ("recon_loss", cl_loss * self.hparams["recon_weight"]),
        ])

    def predict_step(self, test_iter: tf.data.Dataset) -> OrderedDict:
        inputs, _ = test_iter
        cond = inputs["lo_res_inputs"]
        const = inputs["hi_res_inputs"]
        
        if self.hparams["ensemble_size"] is not None:
            noise = [self.noise_gen() for _ in range(self.hparams["ensemble_size"])]
            gen_list = []
            for noise_iter in noise:
                gen_in = [cond] + [const] + [noise_iter]
                gen_iter = self.generator.model(gen_in, training=False)
                gen_list.append(gen_iter)
            gen_out = tf.stack(gen_list, axis=-1)
        else:
            noise = self.noise_gen()
            gen_in = [cond] + [const] + [noise]
            gen_out = self.generator.model(gen_in, training=False)
        return gen_out

    def gradient_penalty(self, real_data, gen_data, cond_data, const_data):
        """
        Calculates gradient penalty based on 'mixture' of generated and ground truth data
        :param real_data: the ground truth (high-res) data
        :param gen_data: the generated/predicted (high-res) data
        :param cond_data: the conditional (low-res) input data of the generator/critic
        :param const_data: the static (high-res) data of the generator/critic
        :return: gradient penalty
        
        NOTE ML: This is now equivalent to the sage of the GradientPenalty-layer in the original WGAN-implementation,
                 cf. https://github.com/ECMWFCode4Earth/tesserugged/blob/561733660f53a2d3beedc55b593ba68ec260040e/dev/gan/dsrnngan/gan.py#L129C67-L129C69
                 and https://github.com/ECMWFCode4Earth/tesserugged/blob/561733660f53a2d3beedc55b593ba68ec260040e/dev/gan/dsrnngan/layers.py#L13
        
        """
        # get mixture of generated and ground truth data
        #shape_dat = (gen_data - real_data).shape
        alpha = tf.random.normal([self.hparams["batch_size"], 1, 1, 1], 0., 1.)
        mix_data = real_data + alpha * (gen_data - real_data)
        critic_in_gen = [cond_data] + [const_data] + [mix_data]

        with tf.GradientTape() as gp_tape:
            gp_tape.watch(mix_data)
            critic_mix = self.critic.model(critic_in_gen, training=True)

        # calculate the gradient on the mixture data...
        grads_mix = gp_tape.gradient(critic_mix, [mix_data])[0]
        # ... and norm it
        norm = tf.sqrt(tf.reduce_mean(tf.square(grads_mix), axis=[1, 2, 3]))
        gp = tf.reduce_mean((norm - 1.) ** 2)

        return gp
    
    @staticmethod
    def critic_loss(real_img, fake_img):
        real_loss = tf.reduce_mean(real_img)
        fake_loss = tf.reduce_mean(fake_img)
        return fake_loss - real_loss


    # Define the loss functions for the generator.
    @staticmethod
    def generator_loss(fake_img):
        return -tf.reduce_mean(fake_img)
    
    
    def save(self, filepath: str, overwrite: bool = True, include_optimizer: bool = True, save_format: str = None,
             signatures=None, options=None, save_traces: bool = True, suffix: str = "_last"):
        """
        Save generator and critic seperately.
        The parameters of this method are equivalent to Keras.model.save ensuring full functionality.
        :param filepath: path to SavedModel or H5 file to save both models
        :param overwrite: Whether to silently overwrite any existing file at the target location, or provide the user
                          with a manual prompt.
        :param include_optimizer: If True, save optimizer's state together.
        :param save_format: Either `'tf'` or `'h5'`, indicating whether to save the model to Tensorflow SavedModel or
                            HDF5. Defaults to 'tf' in TF 2.X, and 'h5' in TF 1.X.
        :param signatures: Signatures to save with the SavedModel. Applicable to the 'tf' format only.
                           Please see the `signatures` argument in `tf.saved_model.save` for details.
        :param options: (only applies to SavedModel format) `tf.saved_model.SaveOptions` object that specifies options
                        for saving to SavedModel.
        :param save_traces: (only applies to SavedModel format) When enabled, the SavedModel will store the function
                            traces for each layer. This can be disabled, so that only the configs of each layer are
                            stored.  Defaults to `True`. Disabling this will decrease
                            serialization time and reduce file size, but it requires that
                            all custom layers/models implement a `get_config()` method.
        :return: -
        """                   
        assert save_format != "h5", f"h5 is not supported as save format for this model"
        
        # save generator and critic seperately
        generator_path, critic_path = Path(filepath).joinpath(f"{self._expname}_generator{suffix}"), \
                                      Path(filepath).joinpath(f"{self._expname}_critic{suffix}")
        
        os.makedirs(generator_path, exist_ok=True)
        os.makedirs(critic_path, exist_ok=True)
        
        if tf.__version__ >= "2.12.0":
            self.generator.save(generator_path, overwrite, save_format)
            self.critic.save(critic_path, overwrite, save_format)
        else:
            self.generator.save(generator_path, overwrite, include_optimizer, save_format, signatures, options, save_traces)
            self.critic.save(critic_path, overwrite, include_optimizer, save_format, signatures, options, save_traces)
        
        # save weights and optimizer state seperately, since the latter is not supported by Keras' save-method due to a bug
        # https://github.com/keras-team/tf-keras/issues/504
        # Note that it also does not work when choosing the h5-format (and when setting include_otimizer = False as in previous TF versions) 
        if include_optimizer:    # required to resume training    
            self.generator.save_weights(generator_path.joinpath(f"{self._expname}_generator{suffix}"), overwrite=overwrite,
                                        save_format=save_format, options=options)
            self.critic.save_weights(critic_path.joinpath(f"{self._expname}_critic{suffix}"), overwrite=overwrite,
                                     save_format=save_format, options=options)
        
            
            generator_opt = generator_path.joinpath(f"{self._expname}_generator_opt{suffix}.pkl")
            critic_opt = critic_path.joinpath(f"{self._expname}_critic_opt{suffix}.pkl")
            
            print(f"Save generator optimiter state to {generator_opt}...")
            save_opt_weights(self.g_optimizer, generator_opt)
            print(f"Save critic optimiter state to {critic_opt}...")
            save_opt_weights(self.c_optimizer, critic_opt)
        


class HarrisWGAN(AbstractModelClass):
    
    def __init__(self, generator: AbstractModelClass, critic: AbstractModelClass, shape_in: List, hparams: dict,
                 varnames_tar: List, savedir: str, expname: str):
        """
        Initialize the HarrisWGANModel class.

        :param generator: The generator model.
        :param critic: The critic model.
        :param shape_in: The input shape of the model. Note: The last two dimensions must denote the number of coarse-grained predictors 
                         and the number of static high-resolution predictors, respectively.
        :param hparams: Dictionary of custom hyperparameters.
        :param varnames_tar: List of target variable names.
        :param savedir: Drectory to save the model.
        :param expname: The name of the experiment.
        """        
        super().__init__(shape_in, hparams, varnames_tar, savedir, expname)

        self.modelname = "harriswgan"
        
        # set hyperparmaters
        self.set_hparams(hparams)
        # set submodels
        self.generator, self.critic = self.set_model(generator, critic)
        # set compile and fit options as well as custom objects
        self.set_compile_options()
        self.set_custom_objects(loss=self.compile_options['loss'])
        self.set_fit_options()
        
    def set_compile_options(self):
        """
        Set compile options for the HarrisWGAN model.
        """
        # set optimizers
        if self.hparams["optimizer"].lower() == "adam":
            optimizer = keras.optimizers.Adam
            kwargs_opt = {"beta_1": 0.0, "beta_2": 0.9}
        elif self.hparams["optimizer"].lower() == "rmsprop":
            optimizer = keras.optimizers.RMSprop
            kwargs_opt = {}
        else:
            raise ValueError("'{0}' is not a valid optimizer. Either choose Adam or RMSprop-optimizer")

        self.optimizer = (optimizer(self.critic.hparams["lr"], **kwargs_opt), optimizer(self.generator.hparams["lr"], **kwargs_opt))
        
    def get_fit_options(self):
        """
        Get options that will be parsed to the fit-method of the Keras model.
        """
        harriswgan_callbacks = []
        
        if self.hparams["lr_decay"]:
            harriswgan_callbacks.append(LearningRateSchedulerHarrisWGAN(self.get_lr_decay(), verbose=1))
        
        if self.hparams["lcheckpointing"]:            
            harriswgan_callbacks.append(ModelCheckpointHarrisWGAN(self._savedir, self._expname, 
                                                                  monitor="val_recon_loss", verbose=1, save_best_only=False, mode="min"))
            
        if self.hparams["learlystopping"]:
            harriswgan_callbacks.append(EarlyStopping(monitor="val_recon_loss", patience=8))
            
        if harriswgan_callbacks is not None:
            return {"callbacks": harriswgan_callbacks}
        else:
            return {}  
        
    def set_model(self, generator, critic):
        """
        Instantiate the generator and critic models and create the HarrisWGAN model instance.
        :param generator: The generator model.
        :param critic: The critic model.
        """
        # get relevant shapes for input and output of generator and critic
        lo_res_in_shp = list(self._input_shape[:-1] )
        hi_res_in_shp = list(np.array(lo_res_in_shp[:2])*int(np.prod(np.array([4,])))) + [self._input_shape[-1]]
        in_noise_shp = lo_res_in_shp[:2] + [self.hparams["noise_channels"]]    
        out_shp = hi_res_in_shp[:2] + [len(self._varnames_tar)]            
        
        shp_gen = {"lo_res_inputs": lo_res_in_shp, "hi_res_inputs": hi_res_in_shp, "noise_input": in_noise_shp}
        # get generator model
        gen_model = generator(shp_gen, self.hparams["hparams_generator"], self._varnames_tar)      
        
        # get critic model
        shp_critic = {"lo_res_inputs": lo_res_in_shp, "hi_res_inputs": hi_res_in_shp, "output": out_shp}
        critc_model = critic(shp_critic, self.hparams["hparams_critic"], self._varnames_tar)
        
        # get hyperparamters of HarrisWGAN only
        hparams_wgan_only = self.hparams.copy()
        hparams_wgan_only.pop("hparams_critic")
        hparams_wgan_only.pop("hparams_generator")
                
        # ...and create HarrisWGAN model instance
        self.model = HarrisWGAN_Model(gen_model, critc_model, hparams_wgan_only, self._expname)

        return gen_model, critc_model
    
        
    def get_lr_decay(self):
        """
        Get callable of learning rate scheduler which can be used as callabck in Keras models.
        Exponential decay is applied to change the learning rate from the start to the end value.
        Note that the exponential decay is calculated based on the learning rate of the generator, but applies to both.
        :return: learning rate scheduler
        
        NOTE SL: taken from wgan_model.py
        """
        decay_st, decay_end = self.hparams["decay_start"], self.hparams["decay_end"]
        lr_start, lr_end = self.hparams["hparams_generator"]["lr"], self.hparams["hparams_generator"]["lr_end"]

        if not decay_end > decay_st:
            raise ValueError("Epoch for end of learning rate decay must be large than start epoch. " +
                             "Your values: {0:d}, {1:d})".format(decay_st, decay_end))

        ne_decay = decay_end - decay_st
        # calculate decay rate from start and end learning rate
        decay_rate = 1./ne_decay*np.log(lr_end/lr_start)

        def lr_scheduler(epoch, lr):
            if epoch < decay_st:
                return lr
            elif decay_st <= epoch < decay_end:
                return lr * tf.math.exp(decay_rate)
            elif epoch >= decay_end:
                return lr

        return lr_scheduler

    def plot_model(self, save_dir, **kwargs):
        """
        Plot generator and critci model separately.
        :param save_dir: directory under which plots will be saved
        :param kwargs: All keyword arguments valid for tf.keras.utils.plot_model
        
        NOTE SL: taken from wgan_model.py
        """
        k_plot_model(self.generator, os.path.join(save_dir, f"plot_{self._expname}_generator.png"), **kwargs)
        k_plot_model(self.critic, os.path.join(save_dir, f"plot_{self._expname}_critic.png"), **kwargs)
    
    
    def load_checkpoint(self, checkpoint_dir, checkpoint_format: str = None):
        """
        Load model from checkpoint that has been either saved with the save-method or with the Checkpoint-callback.
        Requires that the model is compiled!
        :param checkpoint_dir": Base-directory where checkpointed model is saved (must contain generator and critic separately)
        :param checkpoint_format: format of checkpoint, must match the format used for saving.
        """
        generator_path, critic_path = Path(checkpoint_dir).joinpath(f"{self._expname}_generator*"), \
                                      Path(checkpoint_dir).joinpath(f"{self._expname}_critic*")
        
        matching_gen_dir, matching_critic_dir = glob.glob(str(generator_path)), glob.glob(str(critic_path))
        
        if matching_gen_dir:
            generator_path = Path(matching_gen_dir[0])
            suffix_gen = str(generator_path).split("_generator")[-1]
        else:
            raise FileNotFoundError(f"No matching director for generator-model {str(generator_path)} found.")
            
        if matching_critic_dir:
            critic_path = Path(matching_critic_dir[0])
            suffix_critic = str(critic_path).split("_critic")[-1]
        else:
            raise FileNotFoundError(f"No matching director for generator-model {str(critic_path)} found.")
            
        fname_suffix = ".h5" if checkpoint_format == "h5" else ""
        
        p = generator_path.joinpath(f'{self._expname}_generator{suffix_gen}{fname_suffix}')
        print(f"Load weights from {p}")
        self.generator.load_weights(generator_path.joinpath(f"{self._expname}_generator{suffix_gen}{fname_suffix}"))
        self.critic.load_weights(critic_path.joinpath(f"{self._expname}_critic{suffix_critic}{fname_suffix}"))

        opt_gen_path, opt_critic_path = generator_path.joinpath(f"{self._expname}_generator_opt{suffix_gen}.pkl"), \
                                        critic_path.joinpath(f"{self._expname}_critic_opt{suffix_critic}.pkl")
        with open(opt_gen_path, "rb") as f:
            optimizer_weights_gen = pickle.load(f)

        with open(opt_critic_path, "rb") as f:
            optimizer_weights_critic = pickle.load(f)

        # set state for g_optimizer
        self.g_optimizer._create_all_weights(self.generator.trainable_variables)
        self.g_optimizer.set_weights(optimizer_weights_gen)

        # set state for c_optimizer
        self.c_optimizer._create_all_weights(self.critic.trainable_variables)
        self.c_optimizer.set_weights(optimizer_weights_critic)
        
                          
    def set_hparams_default(self):
        """
        Note: Hyperparameter defaults taken from 1) https://github.com/ECMWFCode4Earth/tesserugged/blob/master/dev/gan/dsrnngan/local_config.yaml and 2) https://github.com/ECMWFCode4Earth/tesserugged/blob/master/dev/gan/dsrnngan/models.py
        """
        self.hparams_default = {"batch_size": 2, "nepochs": 30, "lr_decay": False, "decay_start": 3, "decay_end": 20, "stream_mode": "lo_input",
                                "l_embed": False, "ds_steps": [4,], "d_steps": 5, "recon_weight": 1000., "gp_weight": 10., "optimizer": "adam", 
                                "lcheckpointing": True, "learlystopping": False, "recon_loss": "ensmeanMSE", "ensemble_size": 8,  
                                "noise_channels": 4, "hparams_generator": {}, "hparams_critic": {} }
    

class LearningRateSchedulerHarrisWGAN(LearningRateSchedulerWGAN):
    """Note SL: taken from wgan_model.py"""
    def __init__(self, schedule, verbose=0):
        super(LearningRateSchedulerWGAN, self).__init__(schedule, verbose)

    def on_epoch_begin(self, epoch, logs=None):
        if not hasattr(self.model, "g_optimizer"):
            raise AttributeError('Model must have a "g_optimizer" for optimizing the generator.')

        if not hasattr(self.model, "c_optimizer"):
            raise AttributeError('Model must have a "c_optimizer" for optimizing the critic.')

        if not (hasattr(self.model.g_optimizer, "lr") and hasattr(self.model.c_optimizer, "lr")):
            raise ValueError('Optimizer for generator and critic must both have a "lr" attribute.')
        try:  # new API
            lr_g, lr_c = float(K.get_value(self.model.g_optimizer.lr)), \
                         float(K.get_value(self.model.c_optimizer.lr))
            lr_g, lr_c = self.schedule(epoch, lr_g), self.schedule(epoch, lr_c)
        except TypeError:  # Support for old API for backward compatibility
            raise NotImplementedError("WGAN learning rate schedule is not compatible with old API. Update TF Keras.")

        if not (isinstance(lr_g, (tf.Tensor, float, np.float32, np.float64)) and
                isinstance(lr_c, (tf.Tensor, float, np.float32, np.float64))):
            raise ValueError('The output of the "schedule" function '
                             f'should be float. Got: {lr_g} (generator) and {lr_c} (critic)' )
        if isinstance(lr_g, tf.Tensor) and not lr_g.dtype.is_floating \
           and isinstance(lr_c, tf.Tensor) and lr_c.dtype.is_floating:
            raise ValueError(
                f'The dtype of `lr_g` and `lr_c` Tensor should be float. Got: {lr_g.dtype} (generator)'
                f'and {lr_c.dtype} (critic)' )
        # set updated learning rate
        K.set_value(self.model.g_optimizer.lr, K.get_value(lr_g))
        K.set_value(self.model.c_optimizer.lr, K.get_value(lr_c))
        if self.verbose > 0:
            print(f'\nEpoch {epoch + 1}: LearningRateScheduler setting learning '
                  f'rate for generator to {lr_g}, for critic to {lr_c}.')

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        logs['lr_generator'] = K.get_value(self.model.g_optimizer.lr)
        logs['lr_critic'] = K.get_value(self.model.c_optimizer.lr)


class ModelCheckpointHarrisWGAN(ModelCheckpoint):
    """Note SL: taken from wgan_model.py"""
    def __init__(self, filepath, expname, monitor='val_loss', verbose=0, save_best_only=False, save_weights_only=False,
                 mode='auto', save_freq="epoch", options=None, **kwargs):
        super(ModelCheckpointHarrisWGAN, self).__init__(filepath,  monitor, verbose, save_best_only,
                                                  save_weights_only, mode, save_freq, options=options, **kwargs)
        self._expname = expname
        self._save_format = kwargs.pop("save_format", "tf")

    def _save_model(self, epoch, batch, logs):
        """Saves the model.
        ML: The source-code is largely identical to Keras v2.6.0 implementation except that two models,
            the critic and the generator, are saved separately in filepath_gen and filepath_critic (see below).
            Modified source-code is envelopped between 'ML S' and 'ML E'-comment strings.

        Args:
            epoch: the epoch this iteration is in.
            batch: the batch this iteration is in. `None` if the `save_freq`
              is set to `epoch`.
            logs: the `logs` dict passed in to `on_batch_end` or `on_epoch_end`.
        """
        logs = logs or {}

        if isinstance(self.save_freq, int) or self.epochs_since_last_save >= self.period:
            # Block only when saving interval is reached.
            logs = tf_utils.sync_to_numpy_or_python_type(logs)
            self.epochs_since_last_save = 0
            filepath = self._get_file_path(epoch, batch, logs)
            print(f"filepath from _save_model-method: {filepath}")
            # ML S
            if self.save_best_only:
                add_str = "_best"
            else:
                add_str = f"_epoch{epoch+1:05d}"
            
            filepath = Path(filepath).joinpath(f"{self._expname}{add_str}")
            # ML E
            try:
                if self.save_best_only:
                    current = logs.get(self.monitor)
                    if current is None:
                        logging.warning('Can save best model only with %s available, skipping.', self.monitor)
                    else:
                        if self.monitor_op(current, self.best):
                            if self.verbose > 0:
                                print('\nEpoch %05d: %s improved from %0.5f to %0.5f,'
                                      ' saving model to %s' % (epoch + 1, self.monitor,
                                                               self.best, current, filepath))
                            self.best = current
                            
                            # ML S
                            self.model.save(filepath, overwrite=True, include_optimizer=not self.save_weights_only, save_format=self._save_format, 
                                            suffix=add_str)#, options=self._options)
                            # ML E
                        else:
                            if self.verbose > 0:
                                print('\nEpoch %05d: %s did not improve from %0.5f' %
                                      (epoch + 1, self.monitor, self.best))
                else:
                    if self.verbose > 0:
                        print('\nEpoch %05d: saving model to %s' % (epoch + 1, filepath))
                    # ML S
                    self.model.save(filepath, overwrite=True, include_optimizer=not self.save_weights_only, save_format=self._save_format, 
                                    suffix=add_str)#, options=self._options)
                    # ML E
                self._maybe_remove_file()
            except IsADirectoryError as e:  # h5py 3.x
                raise IOError('Please specify a non-directory filepath for'  
                              'ModelCheckpoint. Filepath used is an existing directory: {}'.format(filepath))
            except IOError as e:  # h5py 2.x
                # `e.errno` appears to be `None` so checking the content of `e.args[0]`.
                if 'is a directory' in str(e.args[0]).lower():
                    raise IOError('Please specify a non-directory filepath for '
                                  'ModelCheckpoint. Filepath used is an existing directory: {}'.format(filepath))
                # Re-throw the error for any other causes.
                raise e

####################################################################################
####################################################################################
# some classes and methods that are used in the original Harris GAN implementation
####################################################################################
####################################################################################
class ReflectionPadding2D(Layer):
    def __init__(self, padding=(1, 1), **kwargs):
        self.padding = tuple(padding)
        super(ReflectionPadding2D, self).__init__(**kwargs)

    def compute_output_shape(self, s):
        return (
            s[0],
            None if s[1] is None else s[1]+2*self.padding[0],
            None if s[2] is None else s[2]+2*self.padding[1],
            s[3]
        )

    def call(self, x):
        i_pad, j_pad = self.padding
        return tf.pad(x, [[0, 0], [i_pad, i_pad], [j_pad, j_pad], [0, 0]], 'REFLECT')


class SymmetricPadding2D(Layer):
    def __init__(self, padding=(1, 1), **kwargs):
        self.padding = tuple(padding)
        super(SymmetricPadding2D, self).__init__(**kwargs)

    def compute_output_shape(self, s):
        return (
            s[0],
            None if s[1] is None else s[1]+2*self.padding[0],
            None if s[2] is None else s[2]+2*self.padding[1],
            s[3]
        )

    def call(self, x):
        i_pad, j_pad = self.padding
        return tf.pad(x, [[0, 0], [i_pad, i_pad], [j_pad, j_pad], [0, 0]], 'SYMMETRIC')

In [69]:
@keras.utils.register_keras_serializable(package="Custom", name="Conv2DPadding") 
class Conv2DPadding(Layer):
    def __init__(self, filters, kernel_size, stride, padding, dilations, **kwargs):
        super(Conv2DPadding, self).__init__(**kwargs)
        self.filters = filters
        self.kernel_size = kernel_size
        self.stride = stride
        self.padding = padding
        self.dilations = dilations
        if not isinstance(dilations, int):
            # padding calculation in build() would need to be adjusted to handle a tuple/list
            raise NotImplementedError("Only integer dilation is supported.")
        if padding is None:
            raise ValueError("padding should not be None")

    def build(self, x):
        if self.padding in ('reflect', 'symmetric'):
            pad = tuple((self.dilations*(s-1))//2 for s in self.kernel_size)  # only works if s is odd, or dilation is even
            if self.padding == 'reflect':
                self.padref = ReflectionPadding2D(padding=pad)
            elif self.padding == 'symmetric':
                self.symref = SymmetricPadding2D(padding=pad)
            self.convval = Conv2D(filters=self.filters,
                                  kernel_size=self.kernel_size,
                                  strides=(self.stride, self.stride),
                                  padding='valid',
                                  dilation_rate=self.dilations)
        else:
            self.convsam = Conv2D(filters=self.filters,
                                  kernel_size=self.kernel_size,
                                  strides=(self.stride, self.stride),
                                  padding='same',
                                  dilation_rate=self.dilations)

    def call(self, x):
        if self.padding in ('reflect', 'symmetric'):
            if self.padding == 'reflect':
                x = self.padref(x)
            elif self.padding == 'symmetric':
                x = self.symref(x)
            return self.convval(x)
        else:  # same
            return self.convsam(x)
        
    def get_config(self):
        config = super().get_config()
        config.update({"filters": self.filters,
                  "kernel_size": self.kernel_size,
                  "stride": self.stride, 
                  "padding": self.padding, 
                  "dilations": self.dilations})
        
        return config
        
        
def residual_block(x, filters, conv_size=(3, 3), stride=1, dilations=1, relu_alpha=0.2, padding=None):
    in_channels = int(x.shape[-1])
    x_in = x

    if stride > 1:
        x_in = AveragePooling2D(pool_size=(stride, stride))(x_in)
    if (filters != in_channels):
        x_in = Conv2D(filters=filters, kernel_size=(1, 1))(x_in)

    # first block of activation and 3x3 convolution (possibly strided, although we don't use this)
    x = LeakyReLU(relu_alpha)(x)
    x = Conv2DPadding(filters=filters, kernel_size=conv_size, stride=stride, dilations=dilations, padding=padding)(x)

    # second block of activation and 3x3 unstrided convolution
    x = LeakyReLU(relu_alpha)(x)
    x = Conv2DPadding(filters=filters, kernel_size=conv_size, stride=1, dilations=dilations, padding=padding)(x)
    
    # skip connection
    x = Add()([x, x_in])

    return x


def const_upscale_block(const_input, steps, filters):
    # Map (N x kH x kW x C) to (N x H x W x f), where k is downscaling factor
    const_output = const_input
    for step in steps:
        const_output = Conv2D(filters=filters, kernel_size=(step, step), strides=step, padding="valid", activation="relu")(const_output)
    return const_output


def wasserstein_loss(y_true, y_pred):
    #return 1.
    return K.mean(y_true * y_pred, axis=-1)

def ensmean_MSE(y_true, y_pred):
    pred_mean = tf.squeeze(tf.reduce_mean(y_pred, axis=0), axis=-1)
    y_true_squ = tf.squeeze(y_true, axis=-1)
    return tf.reduce_mean(tf.math.squared_difference(pred_mean, y_true_squ))

def CL_chooser(CLtype):
    if CLtype != "ensmeanMSE":
        raise NotImplementedError(f"{CLtype = } not implemented, only CLtype = 'ensmeanMSE' is available!")
        
    return {
        # "CRPS": sample_crps,
        # "CRPS_phys": sample_crps_phys,
        "ensmeanMSE": ensmean_MSE,
        # "ensmeanMSE_phys": ensmean_MSE_phys#
    }[CLtype]

In [70]:
from unet_model import Sha_UNet, DeepRU_UNet
from wgan_model import WGAN, Critic_Simple
from other_utils import to_list

class ModelEngine(object):
    """
    Class to get and instantiate known models.
    To add new models, please adapt known_models accordingly.
    General info:
    The key value of the implemented model should be based on keras.Model and should expect 'hparams', 'exp_name' and
    'model_savedir' as arguments for initialization. Further keyword arguments are possible.
    For composite models (e.g. GANs):
    The key value should be a tuple whose first element constitutes the composited model (based on keras.Model).
    The following arguments should then be model construction objects to define the components of the composite model.
    Example: WGAN is a composite model consisting of a generator and critic (see wgan_model.py).
             Thus, the derived class should be the first element of the tuple.
             The model constructions of the generator (e.g. a U-Net) and the critic must then constitute the second and
             third element of the tuple, i.e. {"wgan": (WGAN, Sha_UNet, Critic_Simple).
    """

    known_models = {"sha_unet": (Sha_UNet,),
                    "deepru": (DeepRU_UNet,),
                    "sha_wgan": (WGAN, Sha_UNet, Critic_Simple),
                    "harris_wgan": (HarrisWGAN, GeneratorHarris, CriticHarris)}
    
    long_names = ["Sha U-Net", "DeepRU", "Sha WGAN", "Harris WGAN"]
    
    assert len(known_models) == len(long_names), f"Conflicting number of known_models ({len(known_models)})" + \
                                                 f" and long_names ({len(long_names)})."

    def __init__(self, model_name: str):
        """
        Initialize the model if known.
        :param model_name: name of the model (must match any key of self.known_models)
        Hint: Pass help to get an overview of the available models.
        """
        self.modelname = model_name
        self.model = self.known_models[self.modelname]
        self.model_longname = self.long_names[list(self.known_models.keys()).index(self.modelname)]

    def __call__(self, shape_in, varnames_tar, hparams_dict, save_dir, expname, **kwargs):
        """
        Instantiate the model with some required arguments.
        """
        model_list = to_list(self.model)
        target_model = model_list[0]
        model_args = {"shape_in": shape_in, "varnames_tar": varnames_tar, "hparams": hparams_dict,
                      "savedir": save_dir, "expname": expname, **kwargs}

        try:
            if len(model_list) == 1:
                model = target_model(**model_args)
            else:
                submodels = model_list[1:]
                model = target_model(*submodels, **model_args)

                # Fix to ensure that correct modelname is set
                model.modelname = self.modelname
        except Exception as e:
            err_str = str(e)
            raise RuntimeError(f"Failed to instantiate the model. The following error occured: \n {err_str}")

        return model

    @property
    def modelname(self):
        return self._modelname

    @modelname.setter
    def modelname(self, model_name):

        help_str = self._get_help_str()
        model_name_local = model_name.lower()

        if model_name_local in self.known_models.keys():
            self._modelname = model_name_local
        elif model_name_local == "help":
            print(help_str)
            self._modelname = None
        else:
            raise ValueError(f"Model '{model_name}' is unknown. Please specify a known model. {help_str}")

    def _get_help_str(self):
        """
        Create help-string listing all known models.
        """
        return f"Known models are: {', '.join(list(self.known_models.keys()))}"

In [43]:
model_dir = Path("/p/home/jusers/langguth1/juwels/downscaling_maelstrom/downscaling_benchmark/trained_models/t2m/complete/harris_wgan_tests/" +
                 "harris_wgan_recon2000/harris_wgan_recon2000_generator_epoch00003")
data_dir = Path("/p/scratch/deepacf/maelstrom/maelstrom_data/ap5/downscaling_benchmark_dataset/benchmark_t2m/dataset/coarse_input/")
model_basedir = model_dir.parents[0]

dataset="benchmark_t2m"

js_norm = model_basedir.joinpath("norm.json")
model_config_js = model_basedir.joinpath("config_harris_wgan.json")
dataset_config_js = model_basedir.joinpath("config_ds_benchmark_t2m.json")

In [47]:
with open(dataset_config_js) as dsf:
    print(f"Read dataset configuration file '{dataset_config_js}'.")
    ds_dict = js.load(dsf)
    
with open(model_config_js) as dsm:
    print(f"Read dataset configuration file '{model_config_js}'.")
    hparams_dict = js.load(dsm)
    
data_norm = ZScore(ds_dict["norm_dims"])
data_norm.read_norm_from_file(js_norm)

hparams_dict["batch_size"] = 16
ds_dict["batch_size"] = 16

Read dataset configuration file '/p/home/jusers/langguth1/juwels/downscaling_maelstrom/downscaling_benchmark/trained_models/t2m/complete/harris_wgan_tests/harris_wgan_recon2000/config_ds_benchmark_t2m.json'.
Read dataset configuration file '/p/home/jusers/langguth1/juwels/downscaling_maelstrom/downscaling_benchmark/trained_models/t2m/complete/harris_wgan_tests/harris_wgan_recon2000/config_harris_wgan.json'.


In [45]:
#hparams_dict["hparams_critic"] = hparams_dict.pop("hparams_discriminator")

print(hparams_dict)

{'batch_size': 2, 'nepochs': 10, 'lr_decay': True, 'decay_start': 2, 'decay_end': 5, 'stream_mode': 'lo_input', 'l_embed': False, 'ds_steps': [4], 'd_steps': 5, 'recon_weight': 2000.0, 'gp_weight': 10.0, 'optimizer': 'adam', 'lcheckpointing': True, 'learlystopping': False, 'recon_loss': 'ensmeanMSE', 'ensemble_size': 8, 'noise_channels': 4, 'hparams_generator': {'lr': 1e-05, 'lr_end': 1e-06, 'channels_start': 128}, 'hparams_critic': {'lr': 5e-07}}


In [46]:
tfds_train, train_info = prepare_dataset(data_dir, "benchmark_t2m", ds_dict, hparams_dict, "train", norm_obj=data_norm, shuffle=True, lrepeat=True, drop_remainder=True)
tfds_val, val_info = prepare_dataset(data_dir, "benchmark_t2m", ds_dict, hparams_dict, "val", norm_obj=data_norm, shuffle=False, lrepeat=True, drop_remainder=False)

Selected stream mode for train dataset: lo_input
Selected stream mode for val dataset: lo_input


In [49]:
data_norm, shape_in, nsamples, tfds_train_size = train_info["data_norm"], train_info["shape_in"], \
                                                 train_info["nsamples"], train_info["dataset_size"]
ds_obj_train = train_info.get("ds_obj", None)

In [ ]:
# instantiate model...
# Note: Parse varnames from train_info since list of varnames might get updated depending on model and dataset configuration
model_instance = ModelEngine("harris_wgan")
model = model_instance(shape_in, train_info["all_predictands"], hparams_dict,
                       "/p/home/jusers/langguth1/juwels/test_wgan", "test_harris_wgan") 

# ... compile
model.compile(**model.compile_options)

In [ ]:
steps_per_epoch = 5
history = model.fit(x=tfds_train, epochs=3,
                    steps_per_epoch=steps_per_epoch, validation_data=tfds_val, validation_steps=10,
                    verbose=1, **model.fit_options)

In [ ]:
model_savedir = Path("/p/home/jusers/langguth1/juwels/test_wgan")


In [ ]:
model_savedir_last = model_savedir.joinpath(f"test_harris_wgan_last")
model.save(filepath=model_savedir_last, save_format="tf")

In [ ]:
symbolic_weights = getattr(model.g_optimizer, 'weights')
weight_values = K.batch_get_value(symbolic_weights)

weight_values2 = [v.numpy() for v in model.g_optimizer.variables()]
#with open('optimizer.pkl', 'wb') as f:
#    pickle.dump(weight_values, f)

In [ ]:
assert np.all(weight_values[1] == weight_values2[1])

### Test

Initialize new model-object

In [71]:
model_instance_new = ModelEngine("harris_wgan")
model_new = model_instance_new(shape_in, train_info["all_predictands"], hparams_dict,
                               "/p/home/jusers/langguth1/juwels/downscaling_maelstrom/downscaling_benchmark/trained_models/t2m/complete/harris_wgan_tests/harris_wgan_recon2000/harris_wgan_recon2000_epoch00003", "harris_wgan_recon2000") 

model_new.compile(**model_new.compile_options)

### Use load_checkpoint-method

Before using the checkpoint-method to load the model weights and to set the optimizer state, we check their status after compiling:

In [ ]:
print(model_new.g_optimizer.variables())

for var in model_new.generator.trainable_variables:
    print(f"Variable: {var.name}")
    print(f"Weights: {var.numpy()}\n")
    break

In [72]:
model_new.load_checkpoint("/p/home/jusers/langguth1/juwels/downscaling_maelstrom/downscaling_benchmark/trained_models/t2m/complete/harris_wgan_tests/harris_wgan_recon2000/harris_wgan_recon2000_epoch00003", checkpoint_format="tf")

Load weights from /p/home/jusers/langguth1/juwels/downscaling_maelstrom/downscaling_benchmark/trained_models/t2m/complete/harris_wgan_tests/harris_wgan_recon2000/harris_wgan_recon2000_epoch00003/harris_wgan_recon2000_generator_epoch00003/harris_wgan_recon2000_generator_epoch00003


In [54]:
model_new.save("/p/home/jusers/langguth1/juwels/downscaling_maelstrom/downscaling_benchmark/trained_models/t2m/complete/harris_wgan_tests/harris_wgan_recon2000/harris_wgan_recon2000_epoch00003", )

2024-09-19 13:47:22.375864: W tensorflow/python/util/util.cc:348] Sets are not currently considered sequences, but this may change in the future, so consider avoiding using them.


INFO:tensorflow:Assets written to: /p/home/jusers/langguth1/juwels/downscaling_maelstrom/downscaling_benchmark/trained_models/t2m/complete/harris_wgan_tests/harris_wgan_recon2000/harris_wgan_recon2000_epoch00003/harris_wgan_recon2000_generator_last/assets


INFO:tensorflow:Assets written to: /p/home/jusers/langguth1/juwels/downscaling_maelstrom/downscaling_benchmark/trained_models/t2m/complete/harris_wgan_tests/harris_wgan_recon2000/harris_wgan_recon2000_epoch00003/harris_wgan_recon2000_generator_last/assets


/p/software/juwels/stages/2022/software/TensorFlow/2.6.0-gcccoremkl-11.2.0-2021.4.0-CUDA-11.5/lib/python3.9/site-packages/keras/utils/generic_utils.py:494: CustomMaskWarning: Custom mask layers require a config and must override get_config. When loading, the custom mask layer must be passed to the custom_objects argument.
  warnings.warn('Custom mask layers require a config and must override '


INFO:tensorflow:Assets written to: /p/home/jusers/langguth1/juwels/downscaling_maelstrom/downscaling_benchmark/trained_models/t2m/complete/harris_wgan_tests/harris_wgan_recon2000/harris_wgan_recon2000_epoch00003/harris_wgan_recon2000_critic_last/assets


INFO:tensorflow:Assets written to: /p/home/jusers/langguth1/juwels/downscaling_maelstrom/downscaling_benchmark/trained_models/t2m/complete/harris_wgan_tests/harris_wgan_recon2000/harris_wgan_recon2000_epoch00003/harris_wgan_recon2000_critic_last/assets
/p/software/juwels/stages/2022/software/TensorFlow/2.6.0-gcccoremkl-11.2.0-2021.4.0-CUDA-11.5/lib/python3.9/site-packages/keras/utils/generic_utils.py:494: CustomMaskWarning: Custom mask layers require a config and must override get_config. When loading, the custom mask layer must be passed to the custom_objects argument.
  warnings.warn('Custom mask layers require a config and must override '


Save generator optimiter state to /p/home/jusers/langguth1/juwels/downscaling_maelstrom/downscaling_benchmark/trained_models/t2m/complete/harris_wgan_tests/harris_wgan_recon2000/harris_wgan_recon2000_epoch00003/harris_wgan_recon2000_generator_last/harris_wgan_recon2000_generator_opt_last.pkl...
Save critic optimiter state to /p/home/jusers/langguth1/juwels/downscaling_maelstrom/downscaling_benchmark/trained_models/t2m/complete/harris_wgan_tests/harris_wgan_recon2000/harris_wgan_recon2000_epoch00003/harris_wgan_recon2000_critic_last/harris_wgan_recon2000_critic_opt_last.pkl...


In [73]:
print(model_new.g_optimizer.variables())

for var in model_new.generator.trainable_variables:
    print(f"Variable: {var.name}")
    print(f"Weights: {var.numpy()}\n")
    break

[<tf.Variable 'iter:0' shape=() dtype=int64, numpy=12036>, <tf.Variable 'conv2d_105/kernel/m:0' shape=(4, 4, 2, 128) dtype=float32, numpy=
array([[[[ 3.9686535e-02,  9.0638727e-01, -2.6339016e+00, ...,
          -2.0659790e+00,  3.2608819e+00, -2.8059168e+00],
         [-7.7180302e-01,  1.3474488e+00,  7.6417673e-01, ...,
          -7.0788544e-01,  8.3441079e-01, -7.6142305e-01]],

        [[ 8.4358118e-02,  9.0919077e-01, -2.6375849e+00, ...,
          -2.0807047e+00,  3.2312908e+00, -2.8829608e+00],
         [-8.3653700e-01,  1.3369782e+00,  8.7444448e-01, ...,
          -7.2923642e-01,  5.8889490e-01, -8.0537242e-01]],

        [[ 1.2507930e-01,  9.0686858e-01, -2.5998209e+00, ...,
          -2.0765328e+00,  3.2931190e+00, -2.9765441e+00],
         [-8.7164831e-01,  1.2928803e+00,  9.5434904e-01, ...,
          -7.8194040e-01,  5.9354073e-01, -9.7259331e-01]],

        [[ 1.5911113e-01,  9.0029734e-01, -2.6159282e+00, ...,
          -2.0763340e+00,  3.3274121e+00, -3.0202641e+00],
 

### Manually load checkpointed model

In [ ]:
model_new.generator.load_weights("/p/home/jusers/langguth1/juwels/test_wgan/test_wgan_last/test_generator_last/test_generator_last")
model_new.critic.load_weights("/p/home/jusers/langguth1/juwels/test_wgan/test_wgan_last/test_critic_last/test_critic_last")

In [ ]:
with open("/p/home/jusers/langguth1/juwels/test_wgan/test_wgan_last/test_generator_last/test_generator_opt_last.pkl", "rb") as f:
    optimizer_weights_gen = pickle.load(f)
    
with open("/p/home/jusers/langguth1/juwels/test_wgan/test_wgan_last/test_critic_last/test_critic_opt_last.pkl", "rb") as f:
    optimizer_weights_critic = pickle.load(f)

model_new.g_optimizer._create_all_weights(model_new.generator.trainable_variables)
model_new.g_optimizer.set_weights(optimizer_weights_gen)

model_new.c_optimizer._create_all_weights(model_new.critic.trainable_variables)
model_new.c_optimizer.set_weights(optimizer_weights_critic)

In [ ]:
steps_per_epoch = 2
history = model_new.fit(x=tfds_train, epochs=1,
                    steps_per_epoch=steps_per_epoch, validation_data=tfds_val, validation_steps=10,
                    verbose=1, **model.fit_options)

In [ ]:
import os

# Let's assume this is the found directory
found_dir = '/p/home/jusers/langguth1/juwels/downscaling_maelstrom/downscaling_benchmark/trained_models/t2m/lean/sha_wgan_benchmark_t2m/sha_wgan_benchmark_t2m_best'

# Define the base directory (without the suffix)
base_dir = '/p/home/jusers/langguth1/juwels/downscaling_maelstrom/downscaling_benchmark/trained_models/t2m/lean/sha_wgan_benchmark_t2m/sha_wgan_benchmark_t2m'

# Extract the suffix by removing the base directory part
suffix = found_dir[len(base_dir):]  # This will give you the suffix, including the leading '_'

# Optionally, you can remove the leading underscore if you don't want it
suffix = suffix.lstrip('_')



print(f"Suffix: {suffix}")


In [ ]:
model_dir = '/p/home/jusers/langguth1/juwels/downscaling_maelstrom/downscaling_benchmark/trained_models/t2m/lean/sha_wgan_benchmark_t2m/sha_wgan_benchmark_t2m_best/'

suffix = model_dir.split("_")[-1]

print(model_dir)
#print(suffix)
#print(model_dir.replace(f"_{suffix}", f"_generator_{suffix}")) 

In [ ]:
from pathlib import Path

model_dir = Path(model_dir)

expname = model_dir.name
suffix = expname.split("_")[-1]

gen_dir = model_dir.joinpath(expname.replace(suffix, f"generator_{suffix}"))
print(gen_dir)